# 05 Models: LambdaMART, collaborative features, neural ranking

**Purpose.** Train the rankers of PROJECT_PLAN section 5 (item 4) one model family at a time and measure every step against the same references on the same searches. This part sets up the training pipeline, profiles memory and trains the **deployable non-personalised base** (E14). Ablations, collaborative features and neural models follow in later sections.

**Primary benchmark.** Lagged history popularity (`pop_history_lag1d`, history actions at least one day old): 0.337 NDCG@10 on `validation_clean` (E13). Differences below 0.003-0.005 are not treated as meaningful.

**Rules (PROJECT_PLAN 5-6, FINDINGS 2-3).**
- Inputs only from the groups in `features_v3_meta.json`; never `session_id`, `wid`, `label` or `n_candidates`.
- Training searches: the 40,000-search dev subsample without searches with a clamped final gap (post-search actions in the history, E12), without an engaged candidate (every order scores the same) and without single-candidate lists.
- Early stopping on an **inner set** of 10% of these training searches (whole searches, seed 4222), so validation is used only to report, never to pick the number of trees.
- LightGBM LambdaRank with **linear gains** `label_gain=[0,1,2,3]` (LightGBM's default gains are exponential, E06) and NDCG@10 for early stopping; final numbers come from our exact tie-aware evaluator (`scripts/evaluation.py`) on complete validation lists.
- Main population `validation_clean` (clamped searches removed); all validation searches as sensitivity. Model comparisons are paired per-search differences with bootstrap intervals over searches.
- Candidates were shuffled within each search when the features were built (notebook 03), so file order, which encodes the label, cannot reach a model.
- Missing values (unknown metadata) stay missing: LightGBM learns where to send them, instead of treating unknown as "no match".

**How to run.** Locally after notebooks 02 and 03 (they write `data/processed/`). On Colab: open this notebook from GitHub (branch `BRANCH` below); the first cell clones the public repository and mounts Drive. Copy `search_table.parquet`, `split_v1.csv`, `features_v3_meta.json` and the two `features_v3_*.parquet` files to `MyDrive/BT4222 Group Project/processed/` first. CPU is enough for this part.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import os
import sys
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from scipy import sparse

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/models"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository, or update an earlier clone
    import subprocess
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
    if (REPO_ROOT / ".git").is_dir():  # a second run in the same runtime: bring the clone to the branch's latest commit
        subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                        "https://github.com/liamkunert/BT4222-group17-jdsearch.git", str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    PROCESSED = Path("/content/drive/MyDrive/BT4222 Group Project/processed")
    DATA_DIR = Path("/content/drive/MyDrive/BT4222 Group Project/data")
else:
    PROCESSED = REPO_ROOT / "data/processed"
    DATA_DIR = REPO_ROOT / "data"
BEHAVIOUR = DATA_DIR / "user_behavior_data.txt"  # raw records, only for the collaborative features (section 10)

sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import RESULTS, evaluate, log_result, paired_difference, summarise
from scripts.jd_data import SECONDS_PER_DAY, iter_records

pd.options.display.float_format = "{:,.4f}".format
FEATURES = "v3"
SEED = 4222        # data decisions (inner split)
MODEL_SEED = 1     # LightGBM; three seeds only for finalists
meta = json.loads((PROCESSED / f"features_{FEATURES}_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]
ALL_FEATURES = [f for group in meta["groups"].values() for f in group]
assert len(ALL_FEATURES) == len(set(ALL_FEATURES)) == 97
assert not set(ALL_FEATURES) & {"session_id", "wid", "label", "n_candidates"}
print(f"LightGBM {lgb.__version__}; {len(ALL_FEATURES)} features in {len(meta['groups'])} groups; processed data: {PROCESSED}")

LightGBM 4.6.0; 97 features in 13 groups; processed data: C:\Users\liamk\ClaudeProjects\BT4222\BT4222-group17-jdsearch\data\processed


## 1. Training population and inner early-stopping set

In [2]:
dev_ids = assignment.index[(assignment.split == "train") & assignment.dev_subsample]
dev_info = searches.loc[dev_ids]
drops = pd.DataFrame({
    "clamped final gap": dev_info.final_gap_clamped,
    "no engaged candidate": dev_info.n_engaged == 0,
    "single candidate": dev_info.n_candidates == 1,
})
train_pool = dev_ids[~drops.any(axis=1).to_numpy()]
rng = np.random.default_rng(SEED)
inner_ids = np.sort(rng.choice(train_pool.to_numpy(), size=round(0.1 * len(train_pool)), replace=False))
fit_ids = np.setdiff1d(train_pool.to_numpy(), inner_ids)
print(f"dev searches {len(dev_ids):,}; kept for training {len(train_pool):,} "
      f"(fit {len(fit_ids):,}, inner early-stopping set {len(inner_ids):,})")
drops.sum().rename("dropped searches (overlapping)").to_frame()

dev searches 40,000; kept for training 37,567 (fit 33,810, inner early-stopping set 3,757)


,dropped searches (overlapping)
clamped final gap,2116
no engaged candidate,503
single candidate,118


## 2. Loading and memory profile

Features are read **column by column into one pre-allocated float32 matrix** (column-major, which LightGBM accepts without copying), so no intermediate DataFrame of the full table is built. The rows of each search are contiguous in the feature files; LightGBM's `group` is the list of consecutive search sizes.

The profile loads all 97 features for the dev subsample and validation and builds a LightGBM dataset from them, which is what the full-model ablations need. "Peak" is the highest memory the process has used so far.

**Model cache.** Every trained model is saved under `data/processed/models/<key>` (booster at its best iteration, validation scores, summary). The key covers the feature list, LightGBM settings, training and inner searches, feature-file sizes and dates and the LightGBM version, so any change trains a new model; re-running the notebook only trains what is new. `RETRAIN = True` ignores the cache (e.g. before the final runs); on a fresh checkout or on Colab everything is trained.

In [3]:
PROCESS = psutil.Process()


def memory_gb():
    """(current, peak) resident memory of this process in GB."""
    info = PROCESS.memory_info()
    peak = getattr(info, "peak_wset", None)  # Windows
    if peak is None:  # Linux / Colab: ru_maxrss is in KiB
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024
    return info.rss / 1e9, peak / 1e9


LEVELS = ["product", "shop_other", "brand_other", "L3"]
# Derived inputs: sums of v3 columns, built while loading (E19: all action types of the last 7 days pooled).
DERIVED = {f"recent_0_7_{level}": [f"{block}_{level}" for block in ("click_0_7", "cart_0_7", "ord_0_7")]
           for level in LEVELS}
# Inputs stored outside the v3 files (section 10: collaborative features): column -> function(split, version) -> file
# with the same rows in the same order as features_{version}_{split}.parquet.
EXTRA = {}


MEMMAP_GB = 1.5  # larger input matrices go to a memory-mapped file on disk (E23: all training searches)
MEMMAP_DIR = Path("/content/tmp") if IN_COLAB else PROCESSED / "tmp"
memmap_files = []


def feature_file(split, version=None):
    """Feature file of a split; `version` defaults to FEATURES (other versions: e.g. the safety-gap rebuild, E24)."""
    return PROCESSED / f"features_{version or FEATURES}_{split}.parquet"


def load_features(splits, columns, sessions=None, version=None):
    """Session ids, float32 feature matrix (column-major) and labels; rows restricted to `sessions` if given.

    `splits` is one split name or several, whose rows are stacked in that order (every search stays contiguous). A
    column in DERIVED is the sum of its v3 parts (missing if a part is missing); a column in EXTRA is read from its own
    file. Matrices above MEMMAP_GB are memory-mapped, so the operating system can page them out.
    """
    splits = [splits] if isinstance(splits, str) else list(splits)
    selected = []
    for split in splits:
        ids = pq.read_table(feature_file(split, version), columns=["session_id"]).column(0).to_numpy()
        if not (np.diff(ids) >= 0).all():
            raise ValueError("Rows must be sorted by session_id so that every search is contiguous")
        selected.append((split, ids, np.ones(len(ids), dtype=bool) if sessions is None else np.isin(ids, sessions)))
    shape = (sum(int(keep.sum()) for _, _, keep in selected), len(columns))
    if shape[0] * shape[1] * 4 / 1e9 > MEMMAP_GB:
        MEMMAP_DIR.mkdir(parents=True, exist_ok=True)
        memmap_files.append(MEMMAP_DIR / f"inputs_{os.getpid()}_{len(memmap_files)}.npy")
        matrix = np.lib.format.open_memmap(memmap_files[-1], mode="w+", dtype=np.float32, shape=shape, fortran_order=True)
    else:
        matrix = np.empty(shape, dtype=np.float32, order="F")
    offset, labels = 0, []
    for split, ids, keep in selected:
        n = int(keep.sum())
        for j, column in enumerate(columns):
            source = EXTRA[column](split, version) if column in EXTRA else feature_file(split, version)
            parts = DERIVED.get(column, [column])
            matrix[offset:offset + n, j] = sum(pq.read_table(source, columns=[p]).column(0).to_numpy() for p in parts)[keep]
        labels.append(pq.read_table(feature_file(split, version), columns=["label"]).column(0).to_numpy()[keep])
        offset += n
    return np.concatenate([ids[keep] for _, ids, keep in selected]), matrix, np.concatenate(labels)


def remove_memmaps():
    """Delete memory-mapped input files that are no longer in use."""
    for f in list(memmap_files):
        try:
            f.unlink()
            memmap_files.remove(f)
        except (PermissionError, FileNotFoundError):
            pass


def group_sizes(ids):
    """Sizes of consecutive runs of equal session ids (LightGBM's `group`)."""
    bounds = np.r_[0, np.flatnonzero(np.diff(ids)) + 1, len(ids)]
    return np.diff(bounds)

In [4]:
profile = [("start", *memory_gb())]
start = time.time()
train_sid, train_X, train_y = load_features("train_dev", ALL_FEATURES, sessions=train_pool)
profile.append(("train features loaded (97 columns)", *memory_gb()))
categorical = [ALL_FEATURES.index(c) for c in meta["categorical"]]
probe = lgb.Dataset(train_X, label=train_y, group=group_sizes(train_sid), categorical_feature=categorical,
                    free_raw_data=False, params={"verbose": -1, "feature_pre_filter": False}).construct()
profile.append(("LightGBM dataset built", *memory_gb()))
valid_sid, valid_X, valid_y = load_features("validation", ALL_FEATURES)
profile.append(("validation features loaded", *memory_gb()))
seconds = time.time() - start
train_rows, train_gb = train_X.shape[0], train_X.nbytes / 1e9
del probe, train_X, valid_X, train_sid, valid_sid, train_y, valid_y
gc.collect()
profile.append(("all freed", *memory_gb()))
profile = pd.DataFrame(profile, columns=["step", "current_gb", "peak_gb"]).set_index("step")
full_rows = train_rows * (assignment.split == "train").sum() / len(dev_ids)
print(f"training rows {train_rows:,} ({train_gb:.2f} GB as float32); load + build {seconds:.0f} s")
print(f"estimate for all training searches: {full_rows / 1e6:.1f}M rows, "
      f"{full_rows * len(ALL_FEATURES) * 4 / 1e9:.1f} GB as float32 before LightGBM's dataset")
print(f"machine memory: {psutil.virtual_memory().total / 1e9:.1f} GB total")
profile

training rows 3,391,571 (1.32 GB as float32); load + build 17 s
estimate for all training searches: 11.3M rows, 4.4 GB as float32 before LightGBM's dataset
machine memory: 16.9 GB total


,current_gb,peak_gb
step,,
start,0.2529,0.2606
train features loaded (97 columns),1.7990,1.8297
LightGBM dataset built,2.0807,2.0936
validation features loaded,2.7925,2.8086
all freed,0.4585,2.8086


## 3. E14: deployable non-personalised base (G0 + G1 + G2a)

Ten features: query length (G0, acts only through interactions because it is constant within a list), the eight query-title relevance features (G1) and lagged history popularity (G2a). Nothing in the base uses the shopper's history or same-day information, so it is the trained version of the deployable reference.

One fixed LightGBM configuration is used for all ablations (PROJECT_PLAN 5a); it is not tuned here. Success criterion, fixed before running: the lower bound of the paired 95% interval against lagged popularity on `validation_clean` is above +0.003.

In [5]:
PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [10],
    "label_gain": [0, 1, 2, 3],          # linear gains, as in our evaluator
    "lambdarank_truncation_level": 20,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_data_in_leaf": 500,
    "feature_pre_filter": False,
    "seed": MODEL_SEED,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 4,
    "verbose": -1,
}
MAX_ROUNDS, PATIENCE = 3000, 100
BASE = meta["groups"]["G0_search"] + meta["groups"]["G1_relevance"] + meta["groups"]["G2a_popularity_deployable"]


RETRAIN = False  # True: ignore cached models and train everything again
# Bump CACHE_VERSION after changing training or feature-loading code, or meta["categorical"] (not in the key otherwise);
# bump CF_SPEC["version"] after changing the CF code (section 10). Version 1 leaves the existing keys unchanged.
CACHE_VERSION = 1
CACHE = PROCESSED / "models"
FEATURE_FILES = [feature_file(split) for split in ("train_dev", "validation")]
validation_frame = pq.read_table(FEATURE_FILES[1], columns=["session_id", "label"]).to_pandas()


def digest(ids):
    return hashlib.sha1(np.sort(np.asarray(ids, dtype=np.int64)).tobytes()).hexdigest()


def file_stats(f):
    return [f.name, f.stat().st_size, int(f.stat().st_mtime)]


def cache_key(columns, params, fit_sessions, train_splits=("train_dev",), version=None):
    version = version or FEATURES
    spec = {"features": version, "columns": columns, "derived": {c: DERIVED[c] for c in columns if c in DERIVED},
            "params": params, "rounds": [MAX_ROUNDS, PATIENCE], "fit": digest(fit_sessions), "inner": digest(inner_ids),
            "files": [file_stats(feature_file(s, version)) for s in ("train_dev", "validation")],
            "lightgbm": lgb.__version__}
    extra_splits = sorted({*train_splits, "train_dev", "validation"})
    extra = {c: [file_stats(EXTRA[c](s, version)) for s in extra_splits] for c in columns if c in EXTRA}
    if extra:  # only models with extra inputs (CF) get this entry, so earlier keys stay valid
        spec["extra"] = extra
    if CACHE_VERSION != 1:
        spec["cache_version"] = CACHE_VERSION
    if tuple(train_splits) != ("train_dev",):  # likewise only models trained on more than the dev subsample
        spec["train_splits"] = [file_stats(feature_file(s, version)) for s in train_splits]
    return hashlib.sha1(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]


def train_ranker(columns, params=PARAMS, label="", fit_sessions=None, train_splits=("train_dev",), version=None):
    """Fit on the fit searches (or a subset, or searches from several training splits), stop early on the inner set,
    score every validation candidate.

    Returns the booster (best iteration), a summary and the validation scores (session_id, label, score), from the
    cache if this exact model was trained before.
    """
    fit_sessions = fit_ids if fit_sessions is None else fit_sessions
    folder = CACHE / cache_key(columns, params, fit_sessions, train_splits, version)
    if not RETRAIN and (folder / "summary.json").is_file():
        summary = json.loads((folder / "summary.json").read_text())
        return (lgb.Booster(model_file=str(folder / "model.txt")), {**summary, "cached": True},
                validation_frame.assign(score=np.load(folder / "validation_scores.npy")))
    cats = [columns.index(c) for c in meta["categorical"] if c in columns]
    sid, X, y = load_features(train_splits, columns, sessions=fit_sessions, version=version)  # no copies of X
    fit_set = lgb.Dataset(X, label=y, group=group_sizes(sid), feature_name=columns, categorical_feature=cats)
    sid_inner, X_inner, y_inner = load_features("train_dev", columns, sessions=inner_ids, version=version)
    inner_set = lgb.Dataset(X_inner, label=y_inner, group=group_sizes(sid_inner), reference=fit_set)
    start = time.time()
    booster = lgb.train(params, fit_set, num_boost_round=MAX_ROUNDS, valid_sets=[inner_set], valid_names=["inner"],
                        callbacks=[lgb.early_stopping(PATIENCE, verbose=False), lgb.log_evaluation(200)])
    summary = {"model": label, "features": len(columns), "fit_searches": len(fit_sessions),
               "best_iteration": int(booster.best_iteration),
               "inner_ndcg_at_10": float(booster.best_score["inner"]["ndcg@10"]),
               "train_seconds": round(time.time() - start), "peak_memory_gb": round(memory_gb()[1], 2)}
    del X, X_inner, fit_set, inner_set
    gc.collect()
    remove_memmaps()
    sid, X, _ = load_features("validation", columns, version=version)
    assert (sid == validation_frame.session_id.to_numpy()).all()
    scores = booster.predict(X, num_iteration=booster.best_iteration)
    del X
    folder.mkdir(parents=True, exist_ok=True)
    booster.save_model(str(folder / "model.txt"), num_iteration=booster.best_iteration)
    np.save(folder / "validation_scores.npy", scores)
    (folder / "columns.json").write_text(json.dumps(columns))
    (folder / "summary.json").write_text(json.dumps(summary))
    return booster, {**summary, "cached": False}, validation_frame.assign(score=scores)


base_model, base_summary, base_scores = train_ranker(BASE, label="LambdaMART base G0+G1+G2a")
pd.Series(base_summary).to_frame("E14 base")

,E14 base
model,LambdaMART base G0+G1+G2a
features,10
fit_searches,33810
best_iteration,89
inner_ndcg_at_10,0.3447
train_seconds,16
peak_memory_gb,2.7900
cached,True


In [6]:
scored = base_scores.copy()
scored["pop_lag1d"] = pq.read_table(PROCESSED / f"features_{FEATURES}_validation.parquet",
                                    columns=["pop_history_lag1d"]).column(0).to_numpy()
per_search = {
    "LambdaMART base": evaluate(scored, "score", expected_sizes=validation_sizes),
    "lagged history popularity": evaluate(scored.assign(ref=scored.pop_lag1d.fillna(0).astype(float)), "ref",
                                          expected_sizes=validation_sizes),
    "random order": evaluate(scored.assign(constant=0.0), "constant", expected_sizes=validation_sizes),
}
all_ids = per_search["random order"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]

# Sanity check: the reference reproduces its registry value (E13, validation_clean).
registry = pd.read_csv(RESULTS)
logged_ref = registry.query("experiment == 'E13' and model == 'history popularity, lag 1 day' "
                            "and split == 'validation_clean'").ndcg_at_10.iloc[0]
assert abs(per_search["lagged history popularity"].ndcg.loc[clean_ids].mean() - logged_ref) < 1e-4

rows = {}
for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
    for name, per in per_search.items():
        row = summarise(per.loc[ids]).iloc[0].to_dict()
        if name != "lagged history popularity":
            d = paired_difference(per.loc[ids], per_search["lagged history popularity"].loc[ids])
            row.update(vs_lagged_pop=d[0], vs_lagged_pop_low=d[1], vs_lagged_pop_high=d[2])
        rows[(population, name)] = row
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

searches  ndcg_at_10  ndcg_ci_low  \
validation_clean LambdaMART base               18844      0.3470       0.3421   
                 lagged history popularity     18844      0.3374       0.3327   
                 random order                  18844      0.0842       0.0828   
validation (all) LambdaMART base               19772      0.3459       0.3414   
                 lagged history popularity     19772      0.3369       0.3324   
                 random order                  19772      0.0845       0.0830   

                                            ndcg_ci_high  recall_at_10    mrr  \
validation_clean LambdaMART base                  0.3517        0.5345 0.3447   
                 lagged history popularity        0.3423        0.5212 0.3352   
                 random order                     0.0858        0.1587 0.1026   
validation (all) LambdaMART base                  0.3505        0.5342 0.3420   
                 lagged history popularity        0.3414        0.5212 0.3332   
                 random order                     0.0860        0.1599 0.1020   

                                            vs_lagged_pop  vs_lagged_pop_low  \
validation_clean LambdaMART base                   0.0096             0.0076   
                 lagged history popularity            NaN                NaN   
                 random order                     -0.2533            -0.2575   
validation (all) LambdaMART base                   0.0089             0.0071   
                 lagged history popularity            NaN                NaN   
                 random order                     -0.2525            -0.2570   

                                            vs_lagged_pop_high  
validation_clean LambdaMART base                        0.0114  
                 lagged history popularity                 NaN  
                 random order                          -0.2487  
validation (all) LambdaMART base                        0.0107  
                 lagged history popularity                 NaN  
                 random order                          -0.2481

### By segment (validation_clean)

Popularity is much weaker in long, deep-scroll lists (E12), so list length is the segment to watch; activity in the last 7 days separates shoppers with fresh history from returning ones. Each cell is the paired difference base minus lagged popularity within the segment.

In [7]:
info = searches.loc[clean_ids]
segments = {
    "list length": pd.cut(info.n_candidates, [0, 60, 84, 95, 200], labels=["1-60", "61-84", "85-95", "96-200"]),
    "activity in last 7 days": pd.Series(np.where(info.days_since_last_action < 7, "yes", "no"), index=info.index),
}
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search["LambdaMART base"].loc[ids], per_search["lagged history popularity"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "lagged_pop": per_search["lagged history popularity"].ndcg.loc[ids].mean(),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

searches  lagged_pop   base  difference  \
segment                 value                                             
list length             1-60        3811      0.4525 0.4716      0.0191   
                        61-84       1693      0.3383 0.3624      0.0241   
                        85-95       8613      0.3666 0.3735      0.0069   
                        96-200      4727      0.1912 0.1927      0.0015   
activity in last 7 days no          2310      0.3334 0.3421      0.0088   
                        yes        16534      0.3380 0.3477      0.0097   

                                   low   high  
segment                 value                  
list length             1-60    0.0145 0.0238  
                        61-84   0.0174 0.0313  
                        85-95   0.0040 0.0097  
                        96-200 -0.0010 0.0042  
activity in last 7 days no      0.0035 0.0141  
                        yes     0.0078 0.0118

Split importance (share of total gain). This shows what the trees use, not how much shoppers value a signal; correlated features share gain arbitrarily.

In [8]:
importance = pd.Series(base_model.feature_importance("gain"), index=BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

,gain share
pop_history_lag1d,0.8645
bm25,0.0907
title_len,0.0100
query_len,0.0094
title_coverage,0.0084
title_jaccard,0.0058
query_in_brand,0.0047
query_in_cate3,0.0040
title_shared,0.0024
in_catalogue,0.0000


### Log to the results registry

Experiment E14, one row per population. Re-running the notebook does not duplicate rows.

In [9]:
NOTES = (f"LightGBM lambdarank, features {FEATURES} G0+G1+G2a, linear gains, trunc 20, 31 leaves, lr 0.05, "
         f"min leaf 500, inner early stopping (best iteration {base_summary['best_iteration']}), seed {MODEL_SEED}")
logged = pd.read_csv(RESULTS)
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    if ((logged.experiment == "E14") & (logged.model == "LambdaMART base G0+G1+G2a") & (logged.split == split)).any():
        continue
    log_result("E14", "LambdaMART base G0+G1+G2a", split, per_search["LambdaMART base"].loc[ids], notes=NOTES)
pd.read_csv(RESULTS).query("experiment == 'E14'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
41,2026-10-27,E14,LambdaMART base G0+G1+G2a,validation_clean,18844,0.3470,0.3421,0.3517,0.5345,0.3447,282a119,"LightGBM lambdarank, features v3 G0+G1+G2a, li..."
42,2026-10-27,E14,LambdaMART base G0+G1+G2a,validation,19772,0.3459,0.3414,0.3505,0.5342,0.3420,282a119,"LightGBM lambdarank, features v3 G0+G1+G2a, li..."


**Reading the result (E14).**
- The trained base reaches **0.347** NDCG@10 on `validation_clean`, **+0.0096 [0.0076, 0.0114]** over lagged history popularity (0.337); all searches give +0.0089. The success criterion (lower bound above +0.003) is met, so grouping, gains and evaluator behave as expected and the base is the reference for personalisation.
- The gain comes from lists of up to 84 candidates (+0.019 to +0.024); in lists of more than 95 candidates the base is not distinguishable from popularity (+0.0015 [-0.0010, 0.0042]). Activity in the last 7 days makes little difference (+0.009 vs +0.010), as expected for a model without personal history.
- Lagged popularity carries 86% of the split gain, BM25 9%: relevance mainly reorders products of similar popularity. Early stopping chose 89 trees; with ten mostly weak features there is little to learn.
- Memory: the 97 features of the 37,567 training searches take 1.3 GB as float32; with the LightGBM dataset and the validation features the process peaked at 2.8 GB. Full-feature ablations on the dev subsample fit comfortably; all training searches (about 11.3M rows, 4.4 GB before LightGBM's dataset) only if the learning curve asks for them.

## 4. E15: full base with same-day popularity (G0 + G1 + G2a + G2b)

G2b adds the same-day popularity signals: training exposures, the out-of-fold engagement rate and history popularity without the one-day lag. All lists are from one day, so other shoppers' same-day behaviour may come after the search being ranked: the full base is an **optimistic, non-deployable reference**, never a candidate model. It shows how much a model would gain from information it could not have at search time.

In [10]:
def compare(name, against=("LambdaMART base", "lagged history popularity")):
    """Summary of `name` on both populations with paired differences against the references."""
    rows = {}
    for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids)]:
        row = summarise(per_search[name].loc[ids]).iloc[0].to_dict()
        for other in against:
            d = paired_difference(per_search[name].loc[ids], per_search[other].loc[ids])
            row.update({f"vs {other}": d[0], f"vs {other} low": d[1], f"vs {other} high": d[2]})
        rows[population] = row
    table = pd.DataFrame(rows).T
    table["searches"] = table.searches.astype(int)
    return table


def log_once(experiment, model, split, per, summary, version=None):
    """Log one row unless it is already in the registry."""
    logged = pd.read_csv(RESULTS)
    if ((logged.experiment == experiment) & (logged.model == model) & (logged.split == split)).any():
        return
    log_result(experiment, model, split, per,
               notes=(f"LightGBM lambdarank, features {version or FEATURES}, {summary['features']} inputs, "
                      f"{summary['fit_searches']:,} fit searches, inner early stopping "
                      f"(best iteration {summary['best_iteration']}), configuration as E14, seed {MODEL_SEED}"))


FULL_BASE = BASE + meta["groups"]["G2b_popularity_same_day"]
full_base_model, full_base_summary, full_base_scores = train_ranker(FULL_BASE,
                                                                   label="LambdaMART full base G0+G1+G2a+G2b")
per_search["LambdaMART full base"] = evaluate(full_base_scores, "score", expected_sizes=validation_sizes)
display(pd.Series(full_base_summary).to_frame("E15 full base"))
compare("LambdaMART full base")

,E15 full base
model,LambdaMART full base G0+G1+G2a+G2b
features,13
fit_searches,33810
best_iteration,284
inner_ndcg_at_10,0.3646
train_seconds,36
peak_memory_gb,2.7900
cached,True


,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,vs LambdaMART base,vs LambdaMART base low,vs LambdaMART base high,vs lagged history popularity,vs lagged history popularity low,vs lagged history popularity high
validation_clean,18844,0.3655,0.3602,0.3703,0.5589,0.3618,0.0185,0.0164,0.0206,0.0281,0.0253,0.0307
validation (all),19772,0.3647,0.3601,0.3695,0.5596,0.3591,0.0188,0.0166,0.0211,0.0278,0.0254,0.0303


In [11]:
importance = pd.Series(full_base_model.feature_importance("gain"), index=FULL_BASE)
(importance / importance.sum()).sort_values(ascending=False).rename("gain share").to_frame()

,gain share
pop_history,0.7054
pop_engaged_rate,0.1019
bm25,0.0903
pop_exposures,0.0406
pop_history_lag1d,0.0154
title_len,0.0108
title_coverage,0.0091
title_jaccard,0.0090
query_len,0.0065
query_in_brand,0.0062


In [12]:
for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
    log_once("E15", "LambdaMART full base G0+G1+G2a+G2b (same-day, not deployable)", split,
             per_search["LambdaMART full base"].loc[ids], full_base_summary)
pd.read_csv(RESULTS).query("experiment == 'E15'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
43,2026-10-27,E15,"LambdaMART full base G0+G1+G2a+G2b (same-day, ...",validation_clean,18844,0.3655,0.3602,0.3703,0.5589,0.3618,85de125,"LightGBM lambdarank, features v3, 13 inputs, 3..."
44,2026-10-27,E15,"LambdaMART full base G0+G1+G2a+G2b (same-day, ...",validation,19772,0.3647,0.3601,0.3695,0.5596,0.3591,85de125,"LightGBM lambdarank, features v3, 13 inputs, 3..."


**Reading the result (E15).**
- The full base reaches **0.366** on `validation_clean`, **+0.0185 [0.0164, 0.0206]** over the deployable base: about twice the expected gain. As single signals, lagged and unlagged history popularity differ by only 0.004 (0.337 vs 0.341); combined in a model, the same-day signals are worth much more.
- Unlagged history popularity takes 71% of the split gain and the lagged version almost none: where both are available the trees prefer the version that includes same-day events. Which of the three G2b signals carries the gain is not separated here (correlated features share gain).
- The full base stays a non-deployable, optimistic reference: other shoppers' same-day behaviour can come after the search being ranked. It shows that information from the day of the search is valuable, which is exactly what a deployed ranker cannot use.

## 5. E16: learning curve of the deployable full model

The **deployable full model** uses every v3 group except G2b (94 features): the base plus history (H1-H5), context (C1, C2), query-history (Q1) and source (S1) features. It is trained on nested random subsets of the fit searches (10,000, 20,000 and all 33,810; seed 4222), always with the same inner early-stopping set, and scored on the same validation searches.

Two questions: (1) does the model still gain from more data at the size of the dev subsample (decision rule, fixed before running: if all fit searches minus 20,000 is above +0.003 with a paired interval excluding 0, the features for all training searches are built right after the first ablations); (2) as a first look, how much does personal history and context add over the deployable base? The ablations then take this gain apart group by group. All numbers are screening results; the report quotes the finalists retrained on all training searches.

In [13]:
DEPLOYABLE_FULL = [f for f in ALL_FEATURES if f not in meta["groups"]["G2b_popularity_same_day"]]
curve_order = np.random.default_rng(SEED).permutation(fit_ids)
CURVE_SIZES = [10_000, 20_000, len(fit_ids)]
curve_summaries = {}
for n in CURVE_SIZES:
    booster, summary, scores = train_ranker(DEPLOYABLE_FULL, label=f"deployable full, {n:,} fit searches",
                                            fit_sessions=np.sort(curve_order[:n]))
    name = f"deployable full, {n:,} fit searches"
    per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
    curve_summaries[n] = summary
    if n == len(fit_ids):
        full_model = booster
FULL_NAME = f"deployable full, {len(fit_ids):,} fit searches"

rows = []
for i, n in enumerate(CURVE_SIZES):
    name = f"deployable full, {n:,} fit searches"
    per = per_search[name].loc[clean_ids]
    row = {"fit_searches": n, "best_iteration": curve_summaries[n]["best_iteration"],
           "train_seconds": curve_summaries[n]["train_seconds"], "ndcg_at_10": per.ndcg.mean()}
    d = paired_difference(per, per_search["LambdaMART base"].loc[clean_ids])
    row.update(vs_base=d[0], vs_base_low=d[1], vs_base_high=d[2])
    if i:
        previous = per_search[f"deployable full, {CURVE_SIZES[i - 1]:,} fit searches"].loc[clean_ids]
        d = paired_difference(per, previous)
        row.update(vs_previous=d[0], vs_previous_low=d[1], vs_previous_high=d[2])
    rows.append(row)
curve = pd.DataFrame(rows).set_index("fit_searches")
print(f"peak memory so far {memory_gb()[1]:.2f} GB")
curve

peak memory so far 2.81 GB


,best_iteration,train_seconds,ndcg_at_10,vs_base,vs_base_low,vs_base_high,vs_previous,vs_previous_low,vs_previous_high
fit_searches,,,,,,,,,
10000,231,12,0.3841,0.0371,0.0341,0.0402,NaN,NaN,NaN
20000,169,21,0.3853,0.0383,0.0354,0.0412,0.0012,-0.0002,0.0027
33810,361,53,0.3894,0.0423,0.0395,0.0452,0.0041,0.0028,0.0053


In [14]:
compare(FULL_NAME)

,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,vs LambdaMART base,vs LambdaMART base low,vs LambdaMART base high,vs lagged history popularity,vs lagged history popularity low,vs lagged history popularity high
validation_clean,18844,0.3894,0.3844,0.3946,0.5726,0.3953,0.0423,0.0395,0.0452,0.0519,0.0488,0.0551
validation (all),19772,0.4039,0.3987,0.4091,0.5853,0.4081,0.0580,0.0549,0.0613,0.0670,0.0635,0.0704


### Where the deployable full model gains (validation_clean)

Paired difference full model minus deployable base within each segment. Personal history should matter mostly for shoppers active in the last 7 days (E09: personal single signals +0.107 vs +0.027 over random).

In [15]:
rows = []
for segment, labels in segments.items():
    for value in labels.cat.categories if hasattr(labels, "cat") else sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search[FULL_NAME].loc[ids], per_search["LambdaMART base"].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "base": per_search["LambdaMART base"].ndcg.loc[ids].mean(),
                     "full": per_search[FULL_NAME].ndcg.loc[ids].mean(),
                     "difference": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

searches   base   full  difference    low  \
segment                 value                                               
list length             1-60        3811 0.4716 0.5061      0.0344 0.0282   
                        61-84       1693 0.3624 0.4019      0.0394 0.0296   
                        85-95       8613 0.3735 0.4212      0.0477 0.0432   
                        96-200      4727 0.1927 0.2327      0.0400 0.0354   
activity in last 7 days no          2310 0.3421 0.3630      0.0209 0.0150   
                        yes        16534 0.3477 0.3930      0.0453 0.0423   

                                 high  
segment                 value          
list length             1-60   0.0408  
                        61-84  0.0484  
                        85-95  0.0526  
                        96-200 0.0448  
activity in last 7 days no     0.0267  
                        yes    0.0484

Split gain by feature group (share of the total). A description of what the trees use, not a ranking of signals by value: correlated groups share gain arbitrarily, and the ablations, not importances, decide which groups matter.

In [16]:
group_of = {f: g for g, features in meta["groups"].items() for f in features}
importance = pd.Series(full_model.feature_importance("gain"), index=DEPLOYABLE_FULL)
by_group = importance.groupby(importance.index.map(group_of)).sum()
display((by_group / by_group.sum()).sort_values(ascending=False).rename("gain share").to_frame())
(importance / importance.sum()).sort_values(ascending=False).head(15).rename("gain share, top 15 features").to_frame()

,gain share
G2a_popularity_deployable,0.6255
H1_pooled,0.1758
G1_relevance,0.1002
H4_match_recency,0.0388
C2_session_context,0.0103
H2_by_type,0.0102
C1_history_volume,0.0093
H5_metadata_coverage,0.0083
S1_history_source,0.0061
Q1_query_history,0.0056


,"gain share, top 15 features"
pop_history_lag1d,0.6255
pooled_product,0.1639
bm25,0.0678
days_since_shop,0.0211
days_since_product,0.0117
title_len,0.0086
title_jaccard,0.0079
pooled_L3,0.0072
days_since_last_action,0.0062
days_since_brand,0.0060


In [17]:
for n in CURVE_SIZES:
    name = f"deployable full, {n:,} fit searches"
    log_once("E16", f"LambdaMART deployable full (no G2b), {n:,} fit searches", "validation_clean",
             per_search[name].loc[clean_ids], curve_summaries[n])
log_once("E16", f"LambdaMART deployable full (no G2b), {len(fit_ids):,} fit searches", "validation",
         per_search[FULL_NAME].loc[all_ids], curve_summaries[len(fit_ids)])
pd.read_csv(RESULTS).query("experiment == 'E16'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
45,2026-10-27,E16,"LambdaMART deployable full (no G2b), 10,000 fi...",validation_clean,18844,0.3841,0.3791,0.3892,0.5674,0.3897,85de125,"LightGBM lambdarank, features v3, 94 inputs, 1..."
46,2026-10-27,E16,"LambdaMART deployable full (no G2b), 20,000 fi...",validation_clean,18844,0.3853,0.3805,0.3907,0.5677,0.3911,85de125,"LightGBM lambdarank, features v3, 94 inputs, 2..."
47,2026-10-27,E16,"LambdaMART deployable full (no G2b), 33,810 fi...",validation_clean,18844,0.3894,0.3844,0.3946,0.5726,0.3953,85de125,"LightGBM lambdarank, features v3, 94 inputs, 3..."
48,2026-10-27,E16,"LambdaMART deployable full (no G2b), 33,810 fi...",validation,19772,0.4039,0.3987,0.4091,0.5853,0.4081,85de125,"LightGBM lambdarank, features v3, 94 inputs, 3..."


**Reading the result (E16).**
- **Personal history and context add +0.042 [0.040, 0.045]** to the deployable base: 0.389 on `validation_clean` (all searches 0.404, +0.058: the clamped searches inflate it, as expected from E13). Within the expectation (+0.02 to +0.05). The deployable full model is also above the non-deployable full base (0.366).
- The gain is about twice as large for shoppers active in the last 7 days (+0.045 vs +0.021) and, unlike the base, it also holds in long lists (more than 95 candidates: +0.040).
- **Learning curve:** 10,000 to 20,000 fit searches +0.0012 [-0.0002, 0.0027], 20,000 to 33,810 **+0.0041 [0.0028, 0.0053]**. The decision rule fixed before running is met, so the features for all training searches are built right after the first ablations (about 30 Oct). The steps are uneven (best iterations 231, 169, 361 with one seed), so the curve shows that more data still helps, not how much.
- Lagged popularity still takes 63% of the split gain, pooled history 18% (almost all exact-product matches), match recency 4%; context, type and window groups get about 1% each. Importances do not say which groups matter: the ablations decide that.

## 6. E17: untrained "own products first, then popularity" rule

How much of the deployable full model's gain does a simple rule already reach? The rule puts candidates the shopper interacted with before at the top, then sorts by lagged popularity: the obvious way to use history, with no learning. The gap between the rule and the E16 model is what **learned weighting** of history (types, ages, shop and brand evidence, context, query overlap) adds beyond that obvious step.

- **(a) count:** more past actions on the exact product rank higher (`pooled_product`, all types and ages), lagged popularity breaks ties.
- **(b) any match:** any past action on the exact product counts the same, lagged popularity breaks ties.

Popularity is scaled to below 1 and added to the history count, so the order is strictly lexicographic. The audit's 0.386 for a similar rule (E12) was on training rows with the clamped searches included, so it is not comparable.

In [18]:
rule_inputs = pq.read_table(PROCESSED / f"features_{FEATURES}_validation.parquet",
                            columns=["session_id", "pooled_product"]).to_pandas()
assert (rule_inputs.session_id.to_numpy() == scored.session_id.to_numpy()).all()  # same row order as `scored`
own = rule_inputs.pooled_product.fillna(0).to_numpy(dtype=float)
pop = scored.pop_lag1d.fillna(0).to_numpy(dtype=float)
pop_part = pop / (pop.max() + 1)  # strictly below 1: popularity only orders candidates with equal history
RULES = {"rule (a): own-product count, then popularity": own + pop_part,
         "rule (b): any own product, then popularity": (own > 0) + pop_part}
for name, score in RULES.items():
    per_search[name] = evaluate(scored.assign(rule=score), "rule", expected_sizes=validation_sizes)

clamped_ids = all_ids.difference(clean_ids)
shown = ["lagged history popularity", "LambdaMART base", *RULES, FULL_NAME]
table = pd.DataFrame({population: {name: per_search[name].ndcg.loc[ids].mean() for name in shown}
                      for population, ids in [("validation_clean", clean_ids), ("validation (all)", all_ids),
                                              ("clamped only", clamped_ids)]})
pairs = [("rule (a): own-product count, then popularity", "lagged history popularity"),
         ("rule (a): own-product count, then popularity", "LambdaMART base"),
         ("rule (b): any own product, then popularity", "rule (a): own-product count, then popularity"),
         (FULL_NAME, "rule (a): own-product count, then popularity"),
         (FULL_NAME, "rule (b): any own product, then popularity")]
differences = pd.DataFrame(
    {f"{a}  minus  {b}": dict(zip(["difference", "low", "high"],
                                  paired_difference(per_search[a].loc[clean_ids], per_search[b].loc[clean_ids])))
     for a, b in pairs}).T
display(table.rename_axis("NDCG@10"))
differences.rename_axis("validation_clean, paired")

,validation_clean,validation (all),clamped only
NDCG@10,,,
lagged history popularity,0.3374,0.3369,0.3268
LambdaMART base,0.3470,0.3459,0.3232
"rule (a): own-product count, then popularity",0.3733,0.3886,0.6994
"rule (b): any own product, then popularity",0.3709,0.3856,0.6853
"deployable full, 33,810 fit searches",0.3894,0.4039,0.7001


,difference,low,high
"validation_clean, paired",,,
"rule (a): own-product count, then popularity minus lagged history popularity",0.0358,0.0334,0.0382
"rule (a): own-product count, then popularity minus LambdaMART base",0.0263,0.0234,0.0294
"rule (b): any own product, then popularity minus rule (a): own-product count, then popularity",-0.0024,-0.0032,-0.0016
"deployable full, 33,810 fit searches minus rule (a): own-product count, then popularity",0.0161,0.0141,0.0182
"deployable full, 33,810 fit searches minus rule (b): any own product, then popularity",0.0185,0.0165,0.0207


### Where the rule and the trained model differ (validation_clean)

The rule can only differ from popularity in searches whose list contains a product from the shopper's history; everywhere else it **is** lagged popularity. Splitting by that, and by activity in the last 7 days, shows where the trained model gains beyond the rule.

In [19]:
has_own = pd.Series(own > 0).groupby(scored.session_id.to_numpy()).any()
info = searches.loc[clean_ids]
rule_segments = {
    "list contains an own product": pd.Series(np.where(has_own.reindex(clean_ids), "yes", "no"), index=clean_ids),
    "activity in last 7 days": pd.Series(np.where(info.days_since_last_action < 7, "yes", "no"), index=clean_ids),
}
RULE_A = "rule (a): own-product count, then popularity"
rows = []
for segment, labels in rule_segments.items():
    for value in sorted(labels.unique()):
        ids = labels.index[labels == value]
        d = paired_difference(per_search[FULL_NAME].loc[ids], per_search[RULE_A].loc[ids])
        rows.append({"segment": segment, "value": value, "searches": len(ids),
                     "lagged_pop": per_search["lagged history popularity"].ndcg.loc[ids].mean(),
                     "rule_a": per_search[RULE_A].ndcg.loc[ids].mean(),
                     "full_model": per_search[FULL_NAME].ndcg.loc[ids].mean(),
                     "full_minus_rule": d[0], "low": d[1], "high": d[2]})
pd.DataFrame(rows).set_index(["segment", "value"])

searches  lagged_pop  rule_a  full_model  \
segment                      value                                             
list contains an own product no        13784      0.3465  0.3465      0.3638   
                             yes        5060      0.3129  0.4464      0.4589   
activity in last 7 days      no         2310      0.3334  0.3473      0.3630   
                             yes       16534      0.3380  0.3769      0.3930   

                                    full_minus_rule    low   high  
segment                      value                                 
list contains an own product no              0.0174 0.0150 0.0199  
                             yes             0.0126 0.0091 0.0162  
activity in last 7 days      no              0.0157 0.0100 0.0213  
                             yes             0.0161 0.0140 0.0183

In [20]:
logged = pd.read_csv(RESULTS)
entries = [(name, "validation_clean", clean_ids) for name in RULES] + [(RULE_A, "validation", all_ids)]
for name, split, ids in entries:
    if ((logged.experiment == "E17") & (logged.model == name) & (logged.split == split)).any():
        continue
    log_result("E17", name, split, per_search[name].loc[ids],
               notes=f"untrained: sort by pooled_product ({'count' if '(a)' in name else 'any'}), "
                     f"ties by pop_history_lag1d (features {FEATURES})")
pd.read_csv(RESULTS).query("experiment == 'E17'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
49,2026-10-28,E17,"rule (a): own-product count, then popularity",validation_clean,18844,0.3733,0.3685,0.3787,0.5499,0.3800,d11ffe5,"untrained: sort by pooled_product (count), tie..."
50,2026-10-28,E17,"rule (b): any own product, then popularity",validation_clean,18844,0.3709,0.3661,0.3761,0.5498,0.3761,d11ffe5,"untrained: sort by pooled_product (any), ties ..."
51,2026-10-28,E17,"rule (a): own-product count, then popularity",validation,19772,0.3886,0.3838,0.3934,0.5627,0.3937,d11ffe5,"untrained: sort by pooled_product (count), tie..."


**Reading the result (E17).**
- The untrained rule (a) reaches **0.373** on `validation_clean`: +0.036 [0.033, 0.038] over lagged popularity and +0.026 over the trained base. Counting past actions helps a little over "any match" (rule (b) is 0.0024 lower).
- The deployable full model adds **+0.016 [0.014, 0.018]** beyond the rule. Of its +0.052 over lagged popularity, about 70% is the obvious step "known products first" and about 30% is learned (the reading fixed in advance: a rule at 0.38 or above would have left less than 0.01; it did not).
- The learned part comes from both kinds of searches: +0.017 where no candidate is an own product (there the rule *is* lagged popularity, so this is relevance plus shop, brand and category evidence and context) and +0.013 where one is (better weighting of own products against popularity). The ablations take this apart group by group.
- On the clamped searches the rule (0.699) is as good as the trained model (0.700): their own-product matches are post-search actions, which confirms excluding them.
- **Correction (E25, E30):** the 70/30 split used lagged popularity as the reference and the rule lacks relevance features; with own products first and the trained base's score for the rest (section 16), the rule reaches 88% of M1's gain over the base and learning adds +0.0047.

## 7. E18: history representation, pooled vs by type vs by type and window

The three history groups hold the **same evidence at three resolutions** (checked on validation: H3 sums exactly to H2, H2 to H1), with the same four match levels (exact product; shop, brand via other products; L3 category path):

| Model | Inputs | Reads as |
|---|---|---|
| M1 | base + H1 pooled (4) | one pool of history |
| M2 | base + H2 by action type (16) | M2 - M1: does the action type matter? |
| M3 | base + H3 by type and window (28) | M3 - M2: does age within a type matter? |
| M3 + H1 | M3 plus the pooled totals | trees cannot add features up; separates "age does not matter" from "split counts are harder to use" |

Decision rule (fixed before running): a finer representation counts only if it is at least +0.003 better with a paired interval excluding 0; otherwise the simpler one is preferred. Screening results (dev subsample); differences below about 0.003 are within what a different training sample could change.

In [21]:
H = meta["groups"]
model_summaries = {}


def run_models(models):
    """Train (or load) each model and evaluate it on all validation lists; return a summary table."""
    for name, columns in models.items():
        _, summary, scores = train_ranker(columns, label=name)
        per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
        model_summaries[name] = summary
    keys = ["features", "best_iteration", "train_seconds", "cached"]
    table = pd.DataFrame({n: {k: model_summaries[n][k] for k in keys} for n in models}).T
    table["ndcg_clean"] = [per_search[n].ndcg.loc[clean_ids].mean() for n in models]
    return table


def paired_table(pairs, ids=None):
    """Paired per-search differences (A minus B) with bootstrap intervals; validation_clean by default."""
    ids = clean_ids if ids is None else ids
    rows = {}
    for a, b in pairs:
        d = paired_difference(per_search[a].loc[ids], per_search[b].loc[ids])
        rows[f"{a}  minus  {b}"] = {"A": per_search[a].ndcg.loc[ids].mean(), "B": per_search[b].ndcg.loc[ids].mean(),
                                    "difference": d[0], "low": d[1], "high": d[2]}
    return pd.DataFrame(rows).T


M1, M2, M3, M3T = "M1 base + H1 pooled", "M2 base + H2 by type", "M3 base + H3 by type and window", "M3 + H1 totals"
E18_MODELS = {M1: BASE + H["H1_pooled"], M2: BASE + H["H2_by_type"], M3: BASE + H["H3_by_window"],
              M3T: BASE + H["H3_by_window"] + H["H1_pooled"]}
display(run_models(E18_MODELS))
E18_PAIRS = [(M1, "LambdaMART base"), (M2, M1), (M3, M2), (M3, M1), (M3T, M3), (M3T, M2), (FULL_NAME, M3T)]
paired_table(E18_PAIRS)

,features,best_iteration,train_seconds,cached,ndcg_clean
M1 base + H1 pooled,14,372,43,True,0.3879
M2 base + H2 by type,26,328,41,True,0.3879
M3 base + H3 by type and window,38,457,51,True,0.3880
M3 + H1 totals,42,209,32,True,0.3877


,A,B,difference,low,high
M1 base + H1 pooled minus LambdaMART base,0.3879,0.3470,0.0409,0.0380,0.0439
M2 base + H2 by type minus M1 base + H1 pooled,0.3879,0.3879,-0.0000,-0.0011,0.0011
M3 base + H3 by type and window minus M2 base + H2 by type,0.3880,0.3879,0.0001,-0.0011,0.0012
M3 base + H3 by type and window minus M1 base + H1 pooled,0.3880,0.3879,0.0001,-0.0012,0.0014
M3 + H1 totals minus M3 base + H3 by type and window,0.3877,0.3880,-0.0002,-0.0015,0.0009
M3 + H1 totals minus M2 base + H2 by type,0.3877,0.3879,-0.0001,-0.0012,0.0010
"deployable full, 33,810 fit searches minus M3 + H1 totals",0.3894,0.3877,0.0016,0.0006,0.0027


By activity in the last 7 days: shoppers without recent activity have no clicks, so a type or window effect could look different for them.

In [22]:
active = clean_ids[(searches.loc[clean_ids].days_since_last_action < 7).to_numpy()]
inactive = clean_ids.difference(active)
segment_pairs = [(M1, "LambdaMART base"), (M2, M1), (M3, M2), (M3T, M2)]
pd.concat({f"active in last 7 days ({len(active):,})": paired_table(segment_pairs, active),
           f"inactive 7+ days ({len(inactive):,})": paired_table(segment_pairs, inactive)})

A  \
active in last 7 days (16,534) M1 base + H1 pooled  minus  LambdaMART base        0.3915   
                               M2 base + H2 by type  minus  M1 base + H1 pooled   0.3914   
                               M3 base + H3 by type and window  minus  M2 base... 0.3914   
                               M3 + H1 totals  minus  M2 base + H2 by type        0.3913   
inactive 7+ days (2,310)       M1 base + H1 pooled  minus  LambdaMART base        0.3620   
                               M2 base + H2 by type  minus  M1 base + H1 pooled   0.3629   
                               M3 base + H3 by type and window  minus  M2 base... 0.3633   
                               M3 + H1 totals  minus  M2 base + H2 by type        0.3623   

                                                                                       B  \
active in last 7 days (16,534) M1 base + H1 pooled  minus  LambdaMART base        0.3477   
                               M2 base + H2 by type  minus  M1 base + H1 pooled   0.3915   
                               M3 base + H3 by type and window  minus  M2 base... 0.3914   
                               M3 + H1 totals  minus  M2 base + H2 by type        0.3914   
inactive 7+ days (2,310)       M1 base + H1 pooled  minus  LambdaMART base        0.3421   
                               M2 base + H2 by type  minus  M1 base + H1 pooled   0.3620   
                               M3 base + H3 by type and window  minus  M2 base... 0.3629   
                               M3 + H1 totals  minus  M2 base + H2 by type        0.3629   

                                                                                   difference  \
active in last 7 days (16,534) M1 base + H1 pooled  minus  LambdaMART base             0.0438   
                               M2 base + H2 by type  minus  M1 base + H1 pooled       -0.0001   
                               M3 base + H3 by type and window  minus  M2 base...      0.0000   
                               M3 + H1 totals  minus  M2 base + H2 by type            -0.0001   
inactive 7+ days (2,310)       M1 base + H1 pooled  minus  LambdaMART base             0.0198   
                               M2 base + H2 by type  minus  M1 base + H1 pooled        0.0009   
                               M3 base + H3 by type and window  minus  M2 base...      0.0004   
                               M3 + H1 totals  minus  M2 base + H2 by type            -0.0006   

                                                                                      low  \
active in last 7 days (16,534) M1 base + H1 pooled  minus  LambdaMART base         0.0409   
                               M2 base + H2 by type  minus  M1 base + H1 pooled   -0.0013   
                               M3 base + H3 by type and window  minus  M2 base... -0.0011   
                               M3 + H1 totals  minus  M2 base + H2 by type        -0.0012   
inactive 7+ days (2,310)       M1 base + H1 pooled  minus  LambdaMART base         0.0143   
                               M2 base + H2 by type  minus  M1 base + H1 pooled   -0.0019   
                               M3 base + H3 by type and window  minus  M2 base... -0.0027   
                               M3 + H1 totals  minus  M2 base + H2 by type        -0.0032   

                                                                                    high  
active in last 7 days (16,534) M1 base + H1 pooled  minus  LambdaMART base        0.0469  
                               M2 base + H2 by type  minus  M1 base + H1 pooled   0.0010  
                               M3 base + H3 by type and window  minus  M2 base... 0.0011  
                               M3 + H1 totals  minus  M2 base + H2 by type        0.0010  
inactive 7+ days (2,310)       M1 base + H1 pooled  minus  LambdaMART base        0.0254  
                               M2 base + H2 by type  minus  M1 base + H1 pooled   0.0037  
                               M3 base + H3 by type and window  minus  M2 base..

In [23]:
for name in E18_MODELS:
    log_once("E18", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
pd.read_csv(RESULTS).query("experiment == 'E18'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
52,2026-10-28,E18,M1 base + H1 pooled,validation_clean,18844,0.3879,0.3829,0.3932,0.5708,0.3931,74bf284,"LightGBM lambdarank, features v3, 14 inputs, 3..."
53,2026-10-28,E18,M2 base + H2 by type,validation_clean,18844,0.3879,0.3830,0.3934,0.5710,0.3929,74bf284,"LightGBM lambdarank, features v3, 26 inputs, 3..."
54,2026-10-28,E18,M3 base + H3 by type and window,validation_clean,18844,0.3880,0.3828,0.3931,0.5710,0.3930,74bf284,"LightGBM lambdarank, features v3, 38 inputs, 3..."
55,2026-10-28,E18,M3 + H1 totals,validation_clean,18844,0.3877,0.3827,0.3930,0.5705,0.3933,74bf284,"LightGBM lambdarank, features v3, 42 inputs, 3..."


**Reading the result (E18).**
- **Pooled history alone takes almost the whole personal gain:** M1 reaches **0.388**, +0.041 [0.038, 0.044] over the base, against +0.042 for the deployable full model (expected about +0.030).
- **Splitting by action type or age adds nothing:** M2 minus M1 0.0000 [-0.0011, 0.0011], M3 minus M2 +0.0001 [-0.0011, 0.0012]; putting the pooled totals back (M3 + H1) changes nothing either, so it is not the split making the counts harder to use. The same holds for active and inactive shoppers. By the decision rule, **the pooled representation is preferred**.
- Everything else in the full model (match recency, metadata coverage, history volume, session context, query history, history source) adds only **+0.0016 [0.0006, 0.0027]** beyond M3 + H1 at this data size.
- Caveats: the finer representations have more inputs and may need more data (the learning curve still rises, E16), so M1 vs M3 is re-checked in the full-data runs. That broad evidence fades with age as a single signal (E03) does not translate into a gain from telling the model the age.
- **Correction (E25, E30):** with all training data type and window add +0.0013 [0.0002, 0.0025]: detectable, below the threshold, so "adds nothing" overstates; age splits exist only for carts and orders.

## 8. E19: common-window control for the action type

Action type and age are confounded: clicks are only recorded for 7 days, orders mostly come from months ago (FINDINGS 7). A type effect in E18 could therefore be an age effect. Here both models only see the **last 7 days**, where all types are observed:

- **C-a:** base + pooled 0-7 d history, i.e. click + cart + order counts of the last 7 days per match level (`recent_0_7_*`, summed while loading; 4 inputs).
- **C-b:** base + the same evidence split by type (`click_0_7_*`, `cart_0_7_*`, `ord_0_7_*`; 12 inputs).

C-b minus C-a is the type effect without the age confound. Within 7 days orders are rare (0.01% of validation candidates match an ordered product), so this mainly compares clicks with carts. Shoppers without activity in the last 7 days have only zeros here, so the difference is also shown for active shoppers alone.

In [24]:
CA, CB = "C-a base + pooled 0-7 d", "C-b base + by type 0-7 d"
E19_MODELS = {CA: BASE + [f"recent_0_7_{level}" for level in LEVELS],
              CB: BASE + [f"{block}_{level}" for block in ("click_0_7", "cart_0_7", "ord_0_7") for level in LEVELS]}
display(run_models(E19_MODELS))
E19_PAIRS = [(CA, "LambdaMART base"), (CB, "LambdaMART base"), (CB, CA)]
pd.concat({"validation_clean": paired_table(E19_PAIRS),
           f"active in last 7 days ({len(active):,})": paired_table(E19_PAIRS, active)})

,features,best_iteration,train_seconds,cached,ndcg_clean
C-a base + pooled 0-7 d,14,188,27,True,0.3762
C-b base + by type 0-7 d,22,346,61,True,0.3781


A  \
validation_clean               C-a base + pooled 0-7 d  minus  LambdaMART base    0.3762   
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.3781   
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.3781   
active in last 7 days (16,534) C-a base + pooled 0-7 d  minus  LambdaMART base    0.3809   
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.3832   
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.3832   

                                                                                       B  \
validation_clean               C-a base + pooled 0-7 d  minus  LambdaMART base    0.3470   
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.3470   
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.3762   
active in last 7 days (16,534) C-a base + pooled 0-7 d  minus  LambdaMART base    0.3477   
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.3477   
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.3809   

                                                                                   difference  \
validation_clean               C-a base + pooled 0-7 d  minus  LambdaMART base         0.0292   
                               C-b base + by type 0-7 d  minus  LambdaMART base        0.0311   
                               C-b base + by type 0-7 d  minus  C-a base + poo...      0.0019   
active in last 7 days (16,534) C-a base + pooled 0-7 d  minus  LambdaMART base         0.0332   
                               C-b base + by type 0-7 d  minus  LambdaMART base        0.0355   
                               C-b base + by type 0-7 d  minus  C-a base + poo...      0.0023   

                                                                                     low  \
validation_clean               C-a base + pooled 0-7 d  minus  LambdaMART base    0.0270   
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.0287   
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.0009   
active in last 7 days (16,534) C-a base + pooled 0-7 d  minus  LambdaMART base    0.0306   
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.0327   
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.0011   

                                                                                    high  
validation_clean               C-a base + pooled 0-7 d  minus  LambdaMART base    0.0315  
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.0337  
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.0030  
active in last 7 days (16,534) C-a base + pooled 0-7 d  minus  LambdaMART base    0.0358  
                               C-b base + by type 0-7 d  minus  LambdaMART base   0.0382  
                               C-b base + by type 0-7 d  minus  C-a base + poo... 0.0035

In [25]:
for name in E19_MODELS:
    log_once("E19", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
pd.read_csv(RESULTS).query("experiment == 'E19'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
56,2026-10-28,E19,C-a base + pooled 0-7 d,validation_clean,18844,0.3762,0.3711,0.3814,0.5610,0.3805,74bf284,"LightGBM lambdarank, features v3, 14 inputs, 3..."
57,2026-10-28,E19,C-b base + by type 0-7 d,validation_clean,18844,0.3781,0.3731,0.3835,0.5628,0.3826,74bf284,"LightGBM lambdarank, features v3, 22 inputs, 3..."


**Post-hoc (not pre-registered):** what older history adds. M1 pools all ages, C-a only the last 7 days, with the same four match levels; the difference is the value of carts, orders and follows older than 7 days on top of recent evidence.

In [26]:
pd.concat({"validation_clean": paired_table([(M1, CA)]),
           f"active in last 7 days ({len(active):,})": paired_table([(M1, CA)], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(M1, CA)], inactive)})

,,A,B,difference,low,high
validation_clean,M1 base + H1 pooled minus C-a base + pooled 0-7 d,0.3879,0.3762,0.0117,0.0100,0.0134
"active in last 7 days (16,534)",M1 base + H1 pooled minus C-a base + pooled 0-7 d,0.3915,0.3809,0.0106,0.0086,0.0127
"inactive 7+ days (2,310)",M1 base + H1 pooled minus C-a base + pooled 0-7 d,0.3620,0.3427,0.0193,0.0139,0.0250


**Reading the result (E19).**
- Within the last 7 days, splitting by type adds **+0.0019 [0.0009, 0.0030]** (active shoppers +0.0023 [0.0011, 0.0035]): a small effect whose interval excludes 0, but below the +0.003 threshold, so it is not counted as meaningful (expected at most +0.002). With all history (E18) the type adds nothing. **Action type is not an important dimension of history here**, once age is held fixed or all evidence is pooled.
- **Post-hoc, older history matters:** M1 (all ages) beats C-a (last 7 days) by **+0.0117 [0.0100, 0.0134]**, +0.019 for shoppers without recent activity. Carts, orders and follows older than 7 days carry real signal beyond recent evidence; it just does not need to be split by age.
- Together: use **all history, pooled over types and ages**, at four match levels. Which further groups help is the next block.
- **Correction (E25, E30):** C-a has no follows (no 7-day block), so M1 minus C-a also adds follows of every age (0.09% of history).

## 9. E20: remaining feature groups added to pooled history

Starting from M1 (base + pooled history, 0.388), each remaining group is **added on its own** and compared with M1 on the same searches:

| Group | Inputs | Question |
|---|---|---|
| H4 match recency | days since the last action on this product, shop, brand | does it matter *how long ago* the shopper touched it? |
| H5 metadata coverage | share of history with known metadata, per block | can the model discount matches where metadata is often missing? |
| C1 + C2 volume and context | history length and counts per block; time since the last action, actions in the last 7 days, query-less share, last action type | does the shopper's situation change how history is used (interim lead)? |
| time since the last action | `days_since_last_action` alone | the single context signal of the interim lead |
| Q1 query history | overlap of the current query with past queries; matches in query-related history | should history count more when it relates to the query? |
| S1 history source | matches in search-linked vs query-less history | is history from searches better evidence than browsing? |

M1 + all six groups shows their joint value. The plan removed groups from the full model instead; since all of them together added only +0.0016 (E18), removal effects were expected to be small (not a mathematical bound). Decision rule as in E18 (at least +0.003, interval excluding 0). Search-level inputs (context, volume) act only through interactions with candidate features.

In [27]:
DAYS = ["days_since_last_action"]
E20_GROUPS = {"H4 match recency": H["H4_match_recency"], "H5 metadata coverage": H["H5_metadata_coverage"],
              "C1 + C2 volume and context": H["C1_history_volume"] + H["C2_session_context"],
              "time since last action": DAYS, "Q1 query history": H["Q1_query_history"],
              "S1 history source": H["S1_history_source"]}
E20_MODELS = {f"M1 + {group}": BASE + H["H1_pooled"] + columns for group, columns in E20_GROUPS.items()}
M1_ALL = "M1 + all six groups"
E20_MODELS[M1_ALL] = BASE + H["H1_pooled"] + [c for group in E20_GROUPS.values() for c in group if c not in DAYS] + DAYS
assert all(len(set(columns)) == len(columns) for columns in E20_MODELS.values())
display(run_models(E20_MODELS))
E20_PAIRS = [(name, M1) for name in E20_MODELS] + [(FULL_NAME, M1_ALL)]
paired_table(E20_PAIRS)

,features,best_iteration,train_seconds,cached,ndcg_clean
M1 + H4 match recency,17,292,44,True,0.3879
M1 + H5 metadata coverage,21,347,61,True,0.3876
M1 + C1 + C2 volume and context,26,277,53,True,0.3881
M1 + time since last action,15,146,34,True,0.3866
M1 + Q1 query history,22,288,51,True,0.3878
M1 + S1 history source,20,247,43,True,0.3876
M1 + all six groups,50,426,75,True,0.3884


,A,B,difference,low,high
M1 + H4 match recency minus M1 base + H1 pooled,0.3879,0.3879,0.0000,-0.0010,0.0011
M1 + H5 metadata coverage minus M1 base + H1 pooled,0.3876,0.3879,-0.0003,-0.0013,0.0007
M1 + C1 + C2 volume and context minus M1 base + H1 pooled,0.3881,0.3879,0.0002,-0.0008,0.0012
M1 + time since last action minus M1 base + H1 pooled,0.3866,0.3879,-0.0013,-0.0024,-0.0001
M1 + Q1 query history minus M1 base + H1 pooled,0.3878,0.3879,-0.0001,-0.0011,0.0009
M1 + S1 history source minus M1 base + H1 pooled,0.3876,0.3879,-0.0003,-0.0013,0.0007
M1 + all six groups minus M1 base + H1 pooled,0.3884,0.3879,0.0006,-0.0006,0.0018
"deployable full, 33,810 fit searches minus M1 + all six groups",0.3894,0.3884,0.0009,-0.0002,0.0019


By activity in the last 7 days: context should matter most where recent evidence exists (the interim lead was about searches right after the last action).

In [28]:
pd.concat({f"active in last 7 days ({len(active):,})": paired_table([(n, M1) for n in E20_MODELS], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(n, M1) for n in E20_MODELS], inactive)})

A  \
active in last 7 days (16,534) M1 + H4 match recency  minus  M1 base + H1 pooled  0.3915   
                               M1 + H5 metadata coverage  minus  M1 base + H1 ... 0.3910   
                               M1 + C1 + C2 volume and context  minus  M1 base... 0.3914   
                               M1 + time since last action  minus  M1 base + H... 0.3901   
                               M1 + Q1 query history  minus  M1 base + H1 pooled  0.3913   
                               M1 + S1 history source  minus  M1 base + H1 pooled 0.3910   
                               M1 + all six groups  minus  M1 base + H1 pooled    0.3923   
inactive 7+ days (2,310)       M1 + H4 match recency  minus  M1 base + H1 pooled  0.3625   
                               M1 + H5 metadata coverage  minus  M1 base + H1 ... 0.3629   
                               M1 + C1 + C2 volume and context  minus  M1 base... 0.3640   
                               M1 + time since last action  minus  M1 base + H... 0.3617   
                               M1 + Q1 query history  minus  M1 base + H1 pooled  0.3629   
                               M1 + S1 history source  minus  M1 base + H1 pooled 0.3628   
                               M1 + all six groups  minus  M1 base + H1 pooled    0.3608   

                                                                                       B  \
active in last 7 days (16,534) M1 + H4 match recency  minus  M1 base + H1 pooled  0.3915   
                               M1 + H5 metadata coverage  minus  M1 base + H1 ... 0.3915   
                               M1 + C1 + C2 volume and context  minus  M1 base... 0.3915   
                               M1 + time since last action  minus  M1 base + H... 0.3915   
                               M1 + Q1 query history  minus  M1 base + H1 pooled  0.3915   
                               M1 + S1 history source  minus  M1 base + H1 pooled 0.3915   
                               M1 + all six groups  minus  M1 base + H1 pooled    0.3915   
inactive 7+ days (2,310)       M1 + H4 match recency  minus  M1 base + H1 pooled  0.3620   
                               M1 + H5 metadata coverage  minus  M1 base + H1 ... 0.3620   
                               M1 + C1 + C2 volume and context  minus  M1 base... 0.3620   
                               M1 + time since last action  minus  M1 base + H... 0.3620   
                               M1 + Q1 query history  minus  M1 base + H1 pooled  0.3620   
                               M1 + S1 history source  minus  M1 base + H1 pooled 0.3620   
                               M1 + all six groups  minus  M1 base + H1 pooled    0.3620   

                                                                                   difference  \
active in last 7 days (16,534) M1 + H4 match recency  minus  M1 base + H1 pooled      -0.0001   
                               M1 + H5 metadata coverage  minus  M1 base + H1 ...     -0.0005   
                               M1 + C1 + C2 volume and context  minus  M1 base...     -0.0001   
                               M1 + time since last action  minus  M1 base + H...     -0.0015   
                               M1 + Q1 query history  minus  M1 base + H1 pooled      -0.0002   
                               M1 + S1 history source  minus  M1 base + H1 pooled     -0.0005   
                               M1 + all six groups  minus  M1 base + H1 pooled         0.0008   
inactive 7+ days (2,310)       M1 + H4 match recency  minus  M1 base + H1 pooled       0.0005   
                               M1 + H5 metadata coverage  minus  M1 base + H1 ...      0.0010   
                               M1 + C1 + C2 volume and context  minus  M1 base...      0.0020   
                               M1 + time since last action  minus  M1 base + H...     -0.0003   
                               M1 + Q1 query history  minus  M1 base + H1 pooled       0.0009   
                               M1 + S1 history source  minus  M1

In [29]:
for name in E20_MODELS:
    log_once("E20", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
pd.read_csv(RESULTS).query("experiment == 'E20'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
58,2026-10-28,E20,M1 + H4 match recency,validation_clean,18844,0.3879,0.3828,0.3933,0.5711,0.3938,d67925b,"LightGBM lambdarank, features v3, 17 inputs, 3..."
59,2026-10-28,E20,M1 + H5 metadata coverage,validation_clean,18844,0.3876,0.3826,0.3928,0.5710,0.3929,d67925b,"LightGBM lambdarank, features v3, 21 inputs, 3..."
60,2026-10-28,E20,M1 + C1 + C2 volume and context,validation_clean,18844,0.3881,0.3831,0.3934,0.5709,0.3933,d67925b,"LightGBM lambdarank, features v3, 26 inputs, 3..."
61,2026-10-28,E20,M1 + time since last action,validation_clean,18844,0.3866,0.3815,0.3919,0.5692,0.3922,d67925b,"LightGBM lambdarank, features v3, 15 inputs, 3..."
62,2026-10-28,E20,M1 + Q1 query history,validation_clean,18844,0.3878,0.3829,0.3931,0.5700,0.3932,d67925b,"LightGBM lambdarank, features v3, 22 inputs, 3..."
63,2026-10-28,E20,M1 + S1 history source,validation_clean,18844,0.3876,0.3826,0.3928,0.5703,0.3931,d67925b,"LightGBM lambdarank, features v3, 20 inputs, 3..."
64,2026-10-28,E20,M1 + all six groups,validation_clean,18844,0.3885,0.3835,0.3937,0.5714,0.3945,d67925b,"LightGBM lambdarank, features v3, 50 inputs, 3..."


**Reading the result (E20).**
- **No remaining group adds to pooled history.** Against M1 (0.388): match recency 0.0000, metadata coverage -0.0003, volume and context +0.0002, query history -0.0001, history source -0.0003, all six together +0.0006 [-0.0006, 0.0018]; the full model is +0.0009 [-0.0002, 0.0019] above M1 + all six. None meets the decision rule; all intervals include 0 except one.
- **Time since the last action alone slightly hurts** (-0.0013 [-0.0024, -0.0001]): a search-level input can only act through interactions, and here the trees find splits that do not carry over to validation. With the other context inputs it is neutral.
- **The interim leads do not hold up in the trained ranker:** session context does not change how history should be used once the history itself is in the model (also not for active shoppers), and query-related history is no better evidence than all history. In single-signal checks both looked like moderators (FINDINGS 6), but there they stood in for having fresh or related evidence at all, which the pooled counts already capture.
- Working LambdaMART: **base + pooled history (14 inputs)**. Caveats: one training sample of the dev size, so effects below about 0.003 are not resolvable; the full-data runs re-check M1 against M1 + all groups. Query-conditioned weighting of individual history events is tested once more by the attention model.
- **Correction (E25, E30):** effects below 0.003 are not resolvable here; the -0.0013 for time since the last action is most likely early-stopping noise; with all training data M1 + all six adds +0.0020 [0.0010, 0.0031] (section 16), below the threshold.

## 10. E22: item-item collaborative features

All features so far match a candidate against the shopper's **own** history (same product, shop, brand, category). Collaborative filtering adds what **other shoppers** did: two products are similar if many shoppers have both in their history. A candidate the shopper never touched can then still score if it is similar to products they did touch.

**Graph** (PROJECT_PLAN 5b), from training histories only:
- events at least one day before their owner's search (all target searches are on the next day or later), distinct products, the 50 most recent per shopper;
- products with at least 5 shoppers; co-occurrence counts n_ij from one sparse matrix product (never a dense item-by-item matrix), diagonal set to zero (exact repeats stay in H1), pairs with n_ij >= 2;
- similarity n_ij / sqrt(n_i n_j) x n_ij / (n_ij + 10): cosine, shrunk towards 0 for pairs seen by few shoppers; the 50 most similar neighbours per product.

**3-fold cross-fitting.** A training search's own history is in the graph, which would make its similarities look more informative than a validation search's. Training searches therefore get a fold (seeded per search ID), and their features use the graph built from the other two folds; validation uses the graph from all training searches.

**Features per candidate**, over the shopper's distinct history products (all ages): sum, maximum and mean similarity (mean over the shopper's products in the graph) and the number of history products among the candidate's neighbours; plus the number of the shopper's products in the graph (search level). Candidates outside the graph are missing (unknown), not 0. Pooled only: E18 found no gain from splitting history by type or age. Features are cached under `data/processed/cf/`; changing `CF_SPEC` rebuilds them.

In [30]:
CF_SPEC = {"version": "cf1", "min_age_days": 1, "cap": 50, "min_support": 5, "min_pair": 2, "shrink": 10,
           "top_k": 50, "folds": 3}
CF_COLUMNS = ["cf_sum", "cf_max", "cf_mean", "cf_count", "cf_history_in_graph"]
CF_SPLITS = {"train_dev": dev_ids.to_numpy(), "validation": validation_ids.to_numpy(),
             "train_rest": assignment.index[(assignment.split == "train") & ~assignment.dev_subsample].to_numpy()}
CF_TRAINING = {"train_dev", "train_rest"}  # cross-fitted; the other splits use the graph from all training searches
CF_DIR = PROCESSED / "cf"


def cf_name(version=None):
    """CF files of the v3 features are `cf1_*`; other feature versions (E24 safety gap) get their own prefix."""
    version = version or FEATURES
    return CF_SPEC["version"] if version == FEATURES else f"{CF_SPEC['version']}_{version}"


def cf_file(split, version=None):
    return CF_DIR / f"{cf_name(version)}_{split}.parquet"


def read_histories(splits, target_min_age_days=0.0):
    """Graph histories of all training searches (lagged, distinct, most recent first) and target histories.

    Target histories keep the actions at least `target_min_age_days` before their search (0: all; E24: 10 minutes).
    """
    train_set = set(assignment.index[assignment.split == "train"])
    targets = set(np.concatenate([CF_SPLITS[s] for s in splits]).tolist())
    graph, target = {}, {}
    for sid, fields in iter_records(BEHAVIOUR):
        if sid not in train_set and sid not in targets:
            continue
        past = np.array(fields[4].split("_"), dtype=np.int64)
        gaps = np.array(fields[6].split("_"), dtype=np.int64)
        age = np.cumsum(gaps[::-1])[::-1][1:] / SECONDS_PER_DAY
        if sid in train_set:
            recent_first = past[::-1][age[::-1] >= CF_SPEC["min_age_days"]]
            _, first = np.unique(recent_first, return_index=True)
            graph[sid] = recent_first[np.sort(first)]
        if sid in targets:
            target[sid] = np.unique(past[age >= target_min_age_days])
    return graph, target


def lookup(vocab, items):
    """Positions of `items` in the sorted `vocab`, and which items are in it."""
    pos = np.searchsorted(vocab, items)
    found = (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == items)
    return pos, found


def build_graph(item_lists, block=40_000):
    """Sorted product vocabulary and top-k similarity matrix (CSR, one row of neighbours per product).

    The co-occurrence counts are computed in blocks of `block` products (rows), each thresholded and cut to its top-k
    neighbours right away, so the full count matrix never sits in memory.
    """
    items, counts = np.unique(np.concatenate(item_lists), return_counts=True)
    vocab = items[counts >= CF_SPEC["min_support"]]
    rows, cols = [], []
    for u, lst in enumerate(item_lists):
        pos, found = lookup(vocab, lst)
        pos = pos[found][:CF_SPEC["cap"]]  # lists are most recent first
        rows.append(np.full(len(pos), u, np.int32))
        cols.append(pos.astype(np.int32))
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    X = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(item_lists), len(vocab)))
    del rows, cols
    n_i = np.asarray(X.sum(axis=0)).ravel()
    Xt = X.T.tocsr()  # products x shoppers
    k, n = CF_SPEC["top_k"], len(vocab)
    indptr, indices, data, pairs = np.zeros(n + 1, np.int64), [], [], 0
    for lo in range(0, n, block):
        hi = min(lo + block, n)
        co = (Xt[lo:hi] @ X).tocsr()  # co-occurrence counts of products lo..hi with all products
        row_of = np.repeat(np.arange(lo, hi, dtype=np.int32), np.diff(co.indptr))
        co.data[(co.indices == row_of) | (co.data < CF_SPEC["min_pair"])] = 0  # diagonal, rare pairs
        co.eliminate_zeros()
        pairs += co.nnz
        row_of = np.repeat(np.arange(lo, hi, dtype=np.int32), np.diff(co.indptr))
        n_ij = co.data
        sims = (n_ij / np.sqrt(n_i[row_of] * n_i[co.indices]) * n_ij / (n_ij + CF_SPEC["shrink"])).astype(np.float32)
        for r in range(hi - lo):
            a, b = co.indptr[r], co.indptr[r + 1]
            d, c = sims[a:b], co.indices[a:b]
            if b - a > k:
                keep = np.argpartition(-d, k)[:k]
                d, c = d[keep], c[keep]
            indices.append(c)
            data.append(d)
            indptr[lo + r + 1] = indptr[lo + r] + len(c)
        del co, row_of, n_ij, sims
    del X, Xt
    S = sparse.csr_matrix((np.concatenate(data), np.concatenate(indices), indptr), shape=(n, n))
    return vocab, S, {"products": n, "pairs_before_top_k": int(pairs), "neighbours": int(S.nnz)}


def score(out, sid, wids, rows, vocab, S, history, mask):
    """Fill the CF features of one search (rows of `out`) from graph (vocab, S)."""
    hist_pos, hist_found = lookup(vocab, history)
    hist_pos = hist_pos[hist_found]
    out[rows, 4] = len(hist_pos)
    mask[hist_pos] = True
    pos, found = lookup(vocab, wids)
    for r, i in zip(rows[found], pos[found]):
        lo, hi = S.indptr[i], S.indptr[i + 1]
        sims = S.data[lo:hi][mask[S.indices[lo:hi]]]
        out[r, :4] = (sims.sum(), sims.max() if len(sims) else 0.0,
                      sims.sum() / len(hist_pos) if len(hist_pos) else np.nan, len(sims))
    mask[hist_pos] = False

In [31]:
def build_cf_features(version, splits, target_min_age_days=0.0):
    """Build (or keep) the CF feature files of `splits` for a feature version; returns the graph statistics."""
    spec = CF_SPEC if target_min_age_days == 0 else {**CF_SPEC, "target_min_age_days": target_min_age_days}
    spec_file, stats_file = CF_DIR / f"{cf_name(version)}_spec.json", CF_DIR / f"{cf_name(version)}_graphs.json"
    spec_ok = not RETRAIN and spec_file.is_file() and json.loads(spec_file.read_text()) == spec
    missing = [s for s in splits if not (spec_ok and cf_file(s, version).is_file())]  # only these are built
    if missing:
        start = time.time()
        graph_histories, target_histories = read_histories(missing, target_min_age_days)
        cf_fold = {sid: int(np.random.default_rng([SEED, 22, sid]).integers(CF_SPEC["folds"])) for sid in graph_histories}
        print(f"histories read in {time.time() - start:.0f} s: {len(graph_histories):,} graph, {len(target_histories):,} target")
        tables = {split: (ids, np.full((len(ids), len(CF_COLUMNS)), np.nan, dtype=np.float32))
                  for split in missing
                  for ids in [pq.read_table(feature_file(split, version), columns=["session_id", "wid"]).to_pandas()]}
        graph_stats = json.loads(stats_file.read_text()) if spec_ok and stats_file.is_file() else {}
        # one graph per fold for training searches (graph from the other folds), one from all training searches
        training, other = [s for s in missing if s in CF_TRAINING], [s for s in missing if s not in CF_TRAINING]
        jobs = [(f"fold {f}", [s for s in graph_histories if cf_fold[s] != f], training, f)
                for f in range(CF_SPEC["folds"])] if training else []
        if other:
            jobs.append(("all training", list(graph_histories), other, None))
        for name, graph_ids, job_splits, fold in jobs:
            start = time.time()
            vocab, S, stats = build_graph([graph_histories[s] for s in graph_ids])
            mask = np.zeros(len(vocab), dtype=bool)
            for split in job_splits:
                ids, out = tables[split]
                sid, wid = ids.session_id.to_numpy(), ids.wid.to_numpy()
                bounds = np.r_[0, np.flatnonzero(np.diff(sid)) + 1, len(sid)]
                for lo, hi in zip(bounds[:-1], bounds[1:]):
                    if fold is None or cf_fold[sid[lo]] == fold:
                        score(out, sid[lo], wid[lo:hi], np.arange(lo, hi), vocab, S, target_histories[sid[lo]], mask)
            graph_stats[name] = {**stats, "graph_searches": len(graph_ids), "seconds": round(time.time() - start),
                                 "peak_memory_gb": round(memory_gb()[1], 2)}
            del vocab, S, mask
            gc.collect()
        CF_DIR.mkdir(parents=True, exist_ok=True)
        for split, (ids, out) in tables.items():
            assert not np.isnan(out[:, 4]).any()  # every search was scored by exactly one graph
            frame = ids.copy()
            for j, column in enumerate(CF_COLUMNS):
                frame[column] = out[:, j]
            frame.to_parquet(cf_file(split, version), index=False)
        spec_file.write_text(json.dumps(spec))
        stats_file.write_text(json.dumps(graph_stats))
        del graph_histories, target_histories, tables
        gc.collect()
    return json.loads(stats_file.read_text())


graph_stats = build_cf_features(FEATURES, list(CF_SPLITS))
for split in CF_SPLITS:  # the CF rows must line up with the v3 rows
    cf_ids = pq.read_table(cf_file(split), columns=["session_id", "wid"]).to_pandas()
    v3_ids = pq.read_table(feature_file(split), columns=["session_id", "wid"]).to_pandas()
    assert cf_ids.equals(v3_ids)
EXTRA.update({column: cf_file for column in CF_COLUMNS})
pd.DataFrame(graph_stats).T

,products,pairs_before_top_k,neighbours,graph_searches,seconds,peak_memory_gb
fold 0,"263,928.0000","2,850,656.0000","1,574,635.0000","89,359.0000",9.0000,2.8000
fold 1,"265,928.0000","2,835,780.0000","1,566,688.0000","89,206.0000",8.0000,2.8000
fold 2,"263,795.0000","2,835,218.0000","1,557,911.0000","89,097.0000",9.0000,2.8000
all training,"390,532.0000","4,608,524.0000","2,341,008.0000","133,831.0000",19.0000,2.8000


Coverage: share of candidates in the graph (CF features observed) and with at least one similar product in the shopper's history. For training rows also among engaged candidates (training labels only).

In [32]:
rows = {}
for split in ("train_dev", "validation"):
    cf = pq.read_table(cf_file(split), columns=["session_id", "cf_sum", "cf_count"]).to_pandas()
    keep = cf.session_id.isin(train_pool if split == "train_dev" else validation_ids).to_numpy()
    cf = cf[keep]
    rows[split] = {"candidates": len(cf), "in graph": cf.cf_sum.notna().mean(), "similar product in history": (cf.cf_count > 0).mean()}
    if split == "train_dev":
        labels = pq.read_table(FEATURE_FILES[0], columns=["label"]).column(0).to_numpy()[keep]
        engaged = labels >= 1
        rows["train_dev, engaged"] = {"candidates": int(engaged.sum()), "in graph": cf.cf_sum.notna()[engaged].mean(),
                                      "similar product in history": (cf.cf_count[engaged] > 0).mean()}
pd.DataFrame(rows).T

,candidates,in graph,similar product in history
train_dev,"3,391,571.0000",0.1544,0.0225
"train_dev, engaged","71,596.0000",0.4116,0.0925
validation,"1,781,573.0000",0.1867,0.0291


In [33]:
M1_CF = "M1 + CF"
E22_MODELS = {M1_CF: BASE + H["H1_pooled"] + CF_COLUMNS}
display(run_models(E22_MODELS))
pd.concat({"validation_clean": paired_table([(M1_CF, M1), (M1_CF, "LambdaMART base")]),
           f"active in last 7 days ({len(active):,})": paired_table([(M1_CF, M1)], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(M1_CF, M1)], inactive)})

,features,best_iteration,train_seconds,cached,ndcg_clean
M1 + CF,19,439,59,True,0.3898


A  \
validation_clean               M1 + CF  minus  M1 base + H1 pooled 0.3898   
                               M1 + CF  minus  LambdaMART base     0.3898   
active in last 7 days (16,534) M1 + CF  minus  M1 base + H1 pooled 0.3934   
inactive 7+ days (2,310)       M1 + CF  minus  M1 base + H1 pooled 0.3640   

                                                                        B  \
validation_clean               M1 + CF  minus  M1 base + H1 pooled 0.3879   
                               M1 + CF  minus  LambdaMART base     0.3470   
active in last 7 days (16,534) M1 + CF  minus  M1 base + H1 pooled 0.3915   
inactive 7+ days (2,310)       M1 + CF  minus  M1 base + H1 pooled 0.3620   

                                                                    difference  \
validation_clean               M1 + CF  minus  M1 base + H1 pooled      0.0019   
                               M1 + CF  minus  LambdaMART base          0.0428   
active in last 7 days (16,534) M1 + CF  minus  M1 base + H1 pooled      0.0019   
inactive 7+ days (2,310)       M1 + CF  minus  M1 base + H1 pooled      0.0020   

                                                                       low  \
validation_clean               M1 + CF  minus  M1 base + H1 pooled  0.0006   
                               M1 + CF  minus  LambdaMART base      0.0399   
active in last 7 days (16,534) M1 + CF  minus  M1 base + H1 pooled  0.0006   
inactive 7+ days (2,310)       M1 + CF  minus  M1 base + H1 pooled -0.0014   

                                                                     high  
validation_clean               M1 + CF  minus  M1 base + H1 pooled 0.0031  
                               M1 + CF  minus  LambdaMART base     0.0458  
active in last 7 days (16,534) M1 + CF  minus  M1 base + H1 pooled 0.0032  
inactive 7+ days (2,310)       M1 + CF  minus  M1 base + H1 pooled 0.0054

In [34]:
cf_model, _, _ = train_ranker(E22_MODELS[M1_CF], label=M1_CF)  # loaded from the cache
importance = pd.Series(cf_model.feature_importance("gain"), index=E22_MODELS[M1_CF])
display((importance / importance.sum()).sort_values(ascending=False).head(10).rename("gain share").to_frame())
log_once("E22", M1_CF, "validation_clean", per_search[M1_CF].loc[clean_ids], model_summaries[M1_CF])
pd.read_csv(RESULTS).query("experiment == 'E22'")

,gain share
pop_history_lag1d,0.5896
pooled_product,0.1780
bm25,0.0702
cf_sum,0.0426
pooled_shop_other,0.0204
cf_history_in_graph,0.0200
pooled_L3,0.0145
title_jaccard,0.0121
title_len,0.0106
title_coverage,0.0069


,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
65,2026-10-28,E22,M1 + CF,validation_clean,18844,0.3898,0.3847,0.3951,0.5723,0.3952,b455b8b,"LightGBM lambdarank, features v3, 19 inputs, 3..."


**Reading the result (E22).**
- **CF adds +0.0019 [0.0006, 0.0031]** to pooled history (0.3898 vs 0.3879): a small effect whose interval excludes 0, below the +0.003 threshold, so it does not count by the decision rule (expected +0.002 to +0.01). Same for active and inactive shoppers. `cf_sum` takes 4% of the split gain.
- **Why so little: the links are sparse.** The graphs are small (390,532 products, 2.3M neighbour links for all training searches; built in 10-20 s). 15-19% of candidates are in the graph, but only **2-3% have a similar product in the shopper's history** (9% of engaged training candidates). After requiring two shared shoppers per pair, most products have few or no neighbours, so for most candidates CF has nothing to say beyond popularity.
- The time box allows one variant. A denser similarity (truncated SVD of the same history matrix, PROJECT_PLAN 5b add-on) would give every pair of products in the graph a similarity and addresses exactly this sparsity; otherwise CF stops here.

## 11. E23: finalists retrained on all training searches

Everything above was screening on the 40,000-search dev subsample. The finalists are now retrained on **all clean training searches** (125,746: the dev subsample plus `train_rest`, built on Colab in E21), with the same inner early-stopping set and configuration. **These are the numbers for the report.** LightGBM without bagging is deterministic, so one run per model is enough.

Finalists: the deployable base, M1 (base + pooled history, the working model), M3 (history by type and window), M1 + CF and the deployable full model (all 94 deployable inputs). Expected: every model gains a little from more data (the learning curve still rose, E16), and pooled history stays the working model. Inputs above 1.5 GB (M3 and the full model) are loaded into a memory-mapped file, so the 11.3M training rows do not need to fit in free memory.

In [35]:
train_info = searches.loc[assignment.index[assignment.split == "train"]]
full_pool = train_info.index[(~train_info.final_gap_clamped & (train_info.n_engaged > 0)
                              & (train_info.n_candidates > 1)).to_numpy()]
full_fit_ids = np.setdiff1d(full_pool.to_numpy(), inner_ids)
ALL_TRAINING = ("train_dev", "train_rest")
FINALISTS = {"base": (BASE, "LambdaMART base"), "M1": (BASE + H["H1_pooled"], M1),
             "M3": (BASE + H["H3_by_window"], M3), "M1 + CF": (BASE + H["H1_pooled"] + CF_COLUMNS, M1_CF),
             "deployable full": (DEPLOYABLE_FULL, FULL_NAME)}
print(f"training pool {len(full_pool):,} searches; fit {len(full_fit_ids):,}; inner early-stopping set {len(inner_ids):,}")
rows = {}
for short, (columns, dev_name) in FINALISTS.items():
    name = f"{short}, all training"
    _, summary, scores = train_ranker(columns, label=name, fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
    per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
    model_summaries[name] = summary
    d = paired_difference(per_search[name].loc[clean_ids], per_search[dev_name].loc[clean_ids])
    rows[short] = {"inputs": len(columns), "best_iteration": summary["best_iteration"],
                   "train_seconds": summary["train_seconds"], "peak_memory_gb": summary["peak_memory_gb"],
                   "cached": summary["cached"], "dev_trained": per_search[dev_name].ndcg.loc[clean_ids].mean(),
                   "all_training": per_search[name].ndcg.loc[clean_ids].mean(),
                   "all_training_all_searches": per_search[name].ndcg.mean(),
                   "gain_from_data": d[0], "low": d[1], "high": d[2]}
pd.DataFrame(rows).T

training pool 125,746 searches; fit 121,989; inner early-stopping set 3,757


,inputs,best_iteration,train_seconds,peak_memory_gb,cached,dev_trained,all_training,all_training_all_searches,gain_from_data,low,high
base,10,252,120,3.5000,True,0.3470,0.3500,0.3490,0.0029,0.0016,0.0042
M1,14,223,109,3.7300,True,0.3879,0.3887,0.4029,0.0008,-0.0004,0.0020
M3,38,572,214,4.9200,True,0.3880,0.3900,0.4039,0.0021,0.0008,0.0033
M1 + CF,19,432,176,4.9200,True,0.3898,0.3918,0.4060,0.0020,0.0007,0.0031
deployable full,94,441,221,6.7700,True,0.3894,0.3905,0.4051,0.0012,-0.0000,0.0023


The comparisons that carry the report's conclusions, now on the full-data models (paired, `validation_clean`):

In [36]:
F = {short: f"{short}, all training" for short in FINALISTS}
RULE_A = "rule (a): own-product count, then popularity"
E23_PAIRS = [(F["base"], "lagged history popularity"), (F["M1"], F["base"]), (F["M1"], RULE_A),
             (F["M1"], "lagged history popularity"), (F["M3"], F["M1"]), (F["M1 + CF"], F["M1"]),
             (F["deployable full"], F["M1"])]
display(paired_table(E23_PAIRS))
pd.concat({f"active in last 7 days ({len(active):,})": paired_table([(F["M1"], F["base"])], active),
           f"inactive 7+ days ({len(inactive):,})": paired_table([(F["M1"], F["base"])], inactive)})

,A,B,difference,low,high
"base, all training minus lagged history popularity",0.3500,0.3374,0.0125,0.0107,0.0144
"M1, all training minus base, all training",0.3887,0.3500,0.0387,0.0360,0.0415
"M1, all training minus rule (a): own-product count, then popularity",0.3887,0.3733,0.0154,0.0136,0.0175
"M1, all training minus lagged history popularity",0.3887,0.3374,0.0512,0.0480,0.0544
"M3, all training minus M1, all training",0.3900,0.3887,0.0013,0.0002,0.0025
"M1 + CF, all training minus M1, all training",0.3918,0.3887,0.0031,0.0020,0.0043
"deployable full, all training minus M1, all training",0.3905,0.3887,0.0019,0.0008,0.0030


,,A,B,difference,low,high
"active in last 7 days (16,534)","M1, all training minus base, all training",0.3923,0.3507,0.0416,0.0387,0.0446
"inactive 7+ days (2,310)","M1, all training minus base, all training",0.3628,0.3449,0.0179,0.0125,0.0228


In [37]:
for short in FINALISTS:
    for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
        log_once("E23", F[short], split, per_search[F[short]].loc[ids], model_summaries[F[short]])
pd.read_csv(RESULTS).query("experiment == 'E23'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
66,2026-10-28,E23,"base, all training",validation_clean,18844,0.3499,0.3453,0.3547,0.5361,0.3480,aee29bc,"LightGBM lambdarank, features v3, 10 inputs, 1..."
67,2026-10-28,E23,"base, all training",validation,19772,0.3490,0.3445,0.3538,0.5360,0.3455,aee29bc,"LightGBM lambdarank, features v3, 10 inputs, 1..."
68,2026-10-28,E23,"M1, all training",validation_clean,18844,0.3887,0.3836,0.3941,0.5705,0.3944,aee29bc,"LightGBM lambdarank, features v3, 14 inputs, 1..."
69,2026-10-28,E23,"M1, all training",validation,19772,0.4029,0.3979,0.4078,0.5833,0.4066,aee29bc,"LightGBM lambdarank, features v3, 14 inputs, 1..."
70,2026-10-28,E23,"M3, all training",validation_clean,18844,0.3900,0.3848,0.3955,0.5720,0.3957,aee29bc,"LightGBM lambdarank, features v3, 38 inputs, 1..."
71,2026-10-28,E23,"M3, all training",validation,19772,0.4039,0.3989,0.4087,0.5845,0.4077,aee29bc,"LightGBM lambdarank, features v3, 38 inputs, 1..."
72,2026-10-28,E23,"M1 + CF, all training",validation_clean,18844,0.3918,0.3866,0.3973,0.5733,0.3978,aee29bc,"LightGBM lambdarank, features v3, 19 inputs, 1..."
73,2026-10-28,E23,"M1 + CF, all training",validation,19772,0.4060,0.4008,0.4109,0.5861,0.4101,aee29bc,"LightGBM lambdarank, features v3, 19 inputs, 1..."
74,2026-10-28,E23,"deployable full, all training",validation_clean,18844,0.3905,0.3856,0.3959,0.5732,0.3966,aee29bc,"LightGBM lambdarank, features v3, 94 inputs, 1..."
75,2026-10-28,E23,"deployable full, all training",validation,19772,0.4051,0.3999,0.4101,0.5859,0.4094,aee29bc,"LightGBM lambdarank, features v3, 94 inputs, 1..."


**Reading the result (E23).** Numbers for the report (`validation_clean`, all 121,989 fit searches).
- **More data helps only a little:** base +0.0029 [0.0016, 0.0042], M1 +0.0008 (interval includes 0), M3, M1 + CF +0.002, full model +0.0012 (n.s.). Less than the expected +0.003 to +0.006: the E16 learning-curve step (+0.004 from 20k to 33.8k searches) overstated it; with one training sample per size and uneven early stopping, such steps are noisy.
- **The main result holds:** pooled history adds **+0.039 [0.036, 0.042]** to the base (0.389 vs 0.350; active shoppers +0.042, inactive +0.018), and the trained model beats the untrained rule by +0.015. Splitting history by type and window adds +0.0013 [0.0002, 0.0025] and the full model +0.0019: both below the threshold.
- **CF now just meets the decision rule:** M1 + CF **0.392**, +0.0031 [0.0020, 0.0043] over M1 (+0.00308 against the +0.003 threshold). On the dev subsample it was +0.0019; with more training data the model makes slightly better use of the same CF inputs. A small gain at the edge of what we treat as meaningful.
- Deployable base vs lagged popularity: +0.0125 [0.0107, 0.0144]; best model vs lagged popularity about +0.054.
- **Correction (E25, E30):** CF's +0.0031 holds only for this deterministic run; across three seeds CF minus M1 is +0.0004 to +0.0017 (section 16), so CF is no robust gain and "the model uses CF better with more data" is not supported. The E16 step was not "overstated" in a demonstrable way; the later increment was simply smaller.

## 12. E24: robustness, history from at least 10 minutes before the search

In 5% of searches the history contained actions from after the search (the clamped final gap, E12); those searches are excluded, but similar effects could hide in the rest. Here every history action younger than **10 minutes** before the search is dropped, both in the v3 history and context features (`03_features` with `MIN_AGE_DAYS` = 10 minutes, version `v3_minage0.00694444d`, built for `train_dev` and `validation`) and on the shopper's side of the CF features (the graph already uses only actions at least one day old). Popularity and relevance do not change. M1 and M1 + CF are retrained on the dev subsample with these features and compared with their normal versions.

A drop is **not proof of leakage**: actions minutes before a search are legitimate evidence mid-session. The question is whether the personal gain stays clearly positive (expected: a loss of about 0.005-0.015, gain over the base still above about +0.025).

In [38]:
GAP_DAYS = 10 / (24 * 60)
GAP = f"v3_minage{GAP_DAYS:g}d"  # written by 03_features with MIN_AGE_DAYS = 10 minutes
for split in ("train_dev", "validation"):  # same searches, candidates and order as v3; base inputs unchanged
    v3 = pq.read_table(feature_file(split), columns=["session_id", "wid"] + BASE).to_pandas()
    gap = pq.read_table(feature_file(split, GAP), columns=["session_id", "wid"] + BASE).to_pandas()
    assert v3.equals(gap), split
gap_meta = json.loads((PROCESSED / f"features_{GAP}_meta.json").read_text())
assert gap_meta["groups"] == meta["groups"] and abs(gap_meta["min_age_days"] - GAP_DAYS) < 1e-12
gap_stats = build_cf_features(GAP, ["train_dev", "validation"], GAP_DAYS)
for split in ("train_dev", "validation"):
    assert pq.read_table(cf_file(split, GAP), columns=["session_id", "wid"]).to_pandas().equals(
        pq.read_table(feature_file(split), columns=["session_id", "wid"]).to_pandas())
# how much history the gap removes (validation, clean searches)
rows = {}
for name, version in [("v3", None), ("10-minute gap", GAP)]:
    t = pq.read_table(feature_file("validation", version), columns=["session_id", "hist_len", "pooled_product"]).to_pandas()
    t = t[t.session_id.isin(clean_ids)]
    per = t.groupby("session_id").agg(hist_len=("hist_len", "first"), product=("pooled_product", "sum"))
    rows[name] = {"history actions per search": per.hist_len.mean(), "exact-product matches per search": per["product"].mean(),
                  "searches with an exact-product match": (per["product"] > 0).mean()}
pd.DataFrame(rows).T

,history actions per search,exact-product matches per search,searches with an exact-product match
v3,154.3380,1.5819,0.2685
10-minute gap,152.2256,1.2205,0.1943


In [39]:
M1_GAP, M1_CF_GAP = "M1, 10-minute gap", "M1 + CF, 10-minute gap"
E24_MODELS = {M1_GAP: E18_MODELS[M1], M1_CF_GAP: E22_MODELS[M1_CF]}
for name, columns in E24_MODELS.items():
    _, summary, scores = train_ranker(columns, label=name, version=GAP)
    per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
    model_summaries[name] = summary
clamped_ids = all_ids.difference(clean_ids)
E24_PAIRS = [(M1_GAP, M1), (M1_CF_GAP, M1_CF), (M1_GAP, "LambdaMART base"), (M1_CF_GAP, "LambdaMART base"),
             (M1_CF_GAP, M1_GAP)]
pd.concat({"validation_clean": paired_table(E24_PAIRS),
           f"active in last 7 days ({len(active):,})": paired_table(E24_PAIRS[:2], active),
           f"all searches ({len(all_ids):,})": paired_table(E24_PAIRS[:2], all_ids),
           f"clamped only ({len(clamped_ids):,})": paired_table(E24_PAIRS[:2], clamped_ids)})

A  \
validation_clean               M1, 10-minute gap  minus  M1 base + H1 pooled    0.3746   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.3773   
                               M1, 10-minute gap  minus  LambdaMART base        0.3746   
                               M1 + CF, 10-minute gap  minus  LambdaMART base   0.3773   
                               M1 + CF, 10-minute gap  minus  M1, 10-minute gap 0.3773   
active in last 7 days (16,534) M1, 10-minute gap  minus  M1 base + H1 pooled    0.3762   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.3788   
all searches (19,772)          M1, 10-minute gap  minus  M1 base + H1 pooled    0.3785   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.3812   
clamped only (928)             M1, 10-minute gap  minus  M1 base + H1 pooled    0.4592   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.4614   

                                                                                     B  \
validation_clean               M1, 10-minute gap  minus  M1 base + H1 pooled    0.3879   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.3898   
                               M1, 10-minute gap  minus  LambdaMART base        0.3470   
                               M1 + CF, 10-minute gap  minus  LambdaMART base   0.3470   
                               M1 + CF, 10-minute gap  minus  M1, 10-minute gap 0.3746   
active in last 7 days (16,534) M1, 10-minute gap  minus  M1 base + H1 pooled    0.3915   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.3934   
all searches (19,772)          M1, 10-minute gap  minus  M1 base + H1 pooled    0.4020   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.4038   
clamped only (928)             M1, 10-minute gap  minus  M1 base + H1 pooled    0.6896   
                               M1 + CF, 10-minute gap  minus  M1 + CF           0.6880   

                                                                                 difference  \
validation_clean               M1, 10-minute gap  minus  M1 base + H1 pooled        -0.0133   
                               M1 + CF, 10-minute gap  minus  M1 + CF               -0.0125   
                               M1, 10-minute gap  minus  LambdaMART base             0.0275   
                               M1 + CF, 10-minute gap  minus  LambdaMART base        0.0303   
                               M1 + CF, 10-minute gap  minus  M1, 10-minute gap      0.0027   
active in last 7 days (16,534) M1, 10-minute gap  minus  M1 base + H1 pooled        -0.0153   
                               M1 + CF, 10-minute gap  minus  M1 + CF               -0.0146   
all searches (19,772)          M1, 10-minute gap  minus  M1 base + H1 pooled        -0.0235   
                               M1 + CF, 10-minute gap  minus  M1 + CF               -0.0226   
clamped only (928)             M1, 10-minute gap  minus  M1 base + H1 pooled        -0.2303   
                               M1 + CF, 10-minute gap  minus  M1 + CF               -0.2266   

                                                                                    low  \
validation_clean               M1, 10-minute gap  minus  M1 base + H1 pooled    -0.0151   
                               M1 + CF, 10-minute gap  minus  M1 + CF           -0.0144   
                               M1, 10-minute gap  minus  LambdaMART base         0.0251   
                               M1 + CF, 10-minute gap  minus  LambdaMART base    0.0279   
                               M1 + CF, 10-minute gap  minus  M1, 10-minute gap  0.0014   
active in last 7 days (16,534) M1, 10-minute gap  minus  M1 base + H1 pooled    -0.0172   
                               M1 + CF, 10-minute gap  minus  M1 + CF           -0.0169   
all searches (19,772)          M1, 10-minute gap  minu

In [40]:
for name in E24_MODELS:
    log_once("E24", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name], version=GAP)
pd.read_csv(RESULTS).query("experiment == 'E24'")

,date,experiment,model,split,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,commit,notes
76,2026-10-29,E24,"M1, 10-minute gap",validation_clean,18844,0.3746,0.3698,0.3795,0.5594,0.3757,75b9fd8,"LightGBM lambdarank, features v3, 14 inputs, 3..."
77,2026-10-29,E24,"M1 + CF, 10-minute gap",validation_clean,18844,0.3773,0.3725,0.3824,0.5605,0.3793,75b9fd8,"LightGBM lambdarank, features v3, 19 inputs, 3..."


**Reading the result (E24).**
- The gap removes few actions but much exact-product evidence: 2.2 of 154 history actions per validation search, but 23% of the exact-product matches (1.58 to 1.22 per search; searches with such a match 26.9% to 19.4%). Shoppers often look at a product minutes before searching for it.
- **Without them the final model loses 0.0125 [0.0109, 0.0144]** (M1 + CF 0.390 to 0.377 on the dev subsample; M1 -0.0133), at the upper end of the expected 0.005-0.015. **The personal gain stays clearly positive:** M1 + CF with the gap is still **+0.030 [0.028, 0.033]** above the base (was +0.043), M1 +0.028; CF still adds +0.0027. On the clamped searches the drop is -0.23, as expected for post-search actions.
- Reading: about 30% of the personal gain relies on actions from the last 10 minutes. These are legitimate session evidence in a live system, but we cannot rule out that some of them are post-search actions like in the clamped searches; the report should state both. The conclusion (pooled history clearly helps; how and when do not matter) does not depend on them.
- **Correction (E25, E30):** only M1 and M1 + CF were retrained with the gap, so "how and when do not matter" is not tested under it; the pooled-history gain survives (+0.028 to +0.030). The loss comes entirely from searches with a final gap under 10 minutes (section 16).

## 13. E26: CF graph-size check

Training rows get their CF features from a **fold graph** (built on the two other folds, about two thirds of the training searches), validation rows from the graph of **all** training searches (section 10). The larger graph covers more products (19% vs 15% of candidates in the graph), so validation CF features are distributed slightly differently from the ones the models were trained on (independent reviews, E25).

Check without retraining: recompute the validation CF features with each of the three fold graphs (the same graphs as for training rows) and with their average, re-score the trained M1 + CF models (dev subsample and all training searches) and compare with their full-graph scores. Decision rule (fixed before running): if the result moves by 0.002 or more, validation and test CF features come from the fold graphs from now on, also for the neural models.

In [41]:
import warnings

variant_file = CF_DIR / f"{CF_SPEC['version']}_validation_fold_graphs.npz"
val_ids = pq.read_table(feature_file("validation"), columns=["session_id", "wid"]).to_pandas()
if RETRAIN or not variant_file.is_file():
    graph_histories, target_histories = read_histories(["validation"])
    cf_fold = {sid: int(np.random.default_rng([SEED, 22, sid]).integers(CF_SPEC["folds"])) for sid in graph_histories}
    sid, wid = val_ids.session_id.to_numpy(), val_ids.wid.to_numpy()
    bounds = np.r_[0, np.flatnonzero(np.diff(sid)) + 1, len(sid)]
    arrays = {}
    for f in range(CF_SPEC["folds"]):
        vocab, S, _ = build_graph([graph_histories[s] for s in graph_histories if cf_fold[s] != f])
        out = np.full((len(val_ids), len(CF_COLUMNS)), np.nan, dtype=np.float32)
        mask = np.zeros(len(vocab), dtype=bool)
        for lo, hi in zip(bounds[:-1], bounds[1:]):
            score(out, sid[lo], wid[lo:hi], np.arange(lo, hi), vocab, S, target_histories[sid[lo]], mask)
        arrays[f"fold_{f}"] = out
        del vocab, S, mask
    np.savez(variant_file, **arrays)
    del graph_histories, target_histories
    gc.collect()
loaded = np.load(variant_file)
cf_variants = {f"fold graph {f}": loaded[f"fold_{f}"] for f in range(CF_SPEC["folds"])}
with warnings.catch_warnings():  # products in no fold graph stay missing
    warnings.simplefilter("ignore", category=RuntimeWarning)
    cf_variants["average of the fold graphs"] = np.nanmean(np.stack(list(cf_variants.values())), axis=0)
full_cf = pq.read_table(cf_file("validation"), columns=CF_COLUMNS).to_pandas().to_numpy(dtype=np.float32)
pd.DataFrame({name: {"in graph": np.isfinite(a[:, 0]).mean(), "similar product in history": (a[:, 3] > 0).mean(),
                     "mean cf_sum (in graph)": np.nanmean(a[:, 0])}
              for name, a in {"full graph (used so far)": full_cf, **cf_variants}.items()}).T

,in graph,similar product in history,mean cf_sum (in graph)
full graph (used so far),0.1867,0.0291,0.0108
fold graph 0,0.1545,0.0241,0.0111
fold graph 1,0.1549,0.0240,0.0111
fold graph 2,0.1545,0.0242,0.0113
average of the fold graphs,0.1748,0.0326,0.0100


In [42]:
E26_MODELS = {"dev subsample": (M1_CF, M1, {}), "all training": (F["M1 + CF"], F["M1"],
                                                                    {"fit_sessions": full_fit_ids, "train_splits": ALL_TRAINING})}
columns = E22_MODELS[M1_CF]
cf_positions = [columns.index(c) for c in CF_COLUMNS]
_, X_val, _ = load_features("validation", columns)
rows = {}
for model, (name, m1_name, kwargs) in E26_MODELS.items():
    booster, _, cached_scores = train_ranker(columns, label=name, **kwargs)  # loaded from the cache
    assert np.allclose(booster.predict(X_val), cached_scores.score.to_numpy(), atol=1e-9)  # full graph reproduces
    for variant, values in cf_variants.items():
        X = X_val.copy(order="F")
        X[:, cf_positions] = values
        key = f"{name}, CF from {variant}"
        per_search[key] = evaluate(validation_frame.assign(score=booster.predict(X)), "score",
                                   expected_sizes=validation_sizes)
        vs_full = paired_difference(per_search[key].loc[clean_ids], per_search[name].loc[clean_ids])
        vs_m1 = paired_difference(per_search[key].loc[clean_ids], per_search[m1_name].loc[clean_ids])
        rows[(model, variant)] = {"ndcg_clean": per_search[key].ndcg.loc[clean_ids].mean(),
                                  "vs_full_graph": vs_full[0], "low": vs_full[1], "high": vs_full[2],
                                  "vs_M1": vs_m1[0], "vs_M1_low": vs_m1[1], "vs_M1_high": vs_m1[2]}
    rows[(model, "full graph (used so far)")] = {"ndcg_clean": per_search[name].ndcg.loc[clean_ids].mean(),
                                                 **dict(zip(["vs_M1", "vs_M1_low", "vs_M1_high"], paired_difference(
                                                     per_search[name].loc[clean_ids], per_search[m1_name].loc[clean_ids])))}
del X_val
pd.DataFrame(rows).T

ndcg_clean  vs_full_graph     low  \
dev subsample fold graph 0                    0.3890        -0.0008 -0.0016   
              fold graph 1                    0.3891        -0.0007 -0.0014   
              fold graph 2                    0.3893        -0.0005 -0.0013   
              average of the fold graphs      0.3897        -0.0001 -0.0007   
              full graph (used so far)        0.3898            NaN     NaN   
all training  fold graph 0                    0.3913        -0.0004 -0.0012   
              fold graph 1                    0.3915        -0.0003 -0.0009   
              fold graph 2                    0.3913        -0.0004 -0.0011   
              average of the fold graphs      0.3918         0.0001 -0.0004   
              full graph (used so far)        0.3918            NaN     NaN   

                                            high  vs_M1  vs_M1_low  vs_M1_high  
dev subsample fold graph 0               -0.0001 0.0011    -0.0002      0.0023  
              fold graph 1                0.0001 0.0012    -0.0000      0.0025  
              fold graph 2                0.0002 0.0014     0.0002      0.0026  
              average of the fold graphs  0.0005 0.0018     0.0006      0.0031  
              full graph (used so far)       NaN 0.0019     0.0006      0.0031  
all training  fold graph 0                0.0003 0.0026     0.0015      0.0039  
              fold graph 1                0.0004 0.0028     0.0017      0.0040  
              fold graph 2                0.0003 0.0027     0.0015      0.0038  
              average of the fold graphs  0.0006 0.0031     0.0020      0.0043  
              full graph (used so far)       NaN 0.0031     0.0020      0.0043

**Reading the result (E26).**
- Fold graphs cover fewer validation candidates (15.5% vs 18.7% in the graph), as on training rows. Re-scoring the trained M1 + CF models with fold-graph CF features changes `validation_clean` by **-0.0003 to -0.0008** (dev subsample and all training searches; intervals touching 0); the average of the three fold graphs gives the same result as the full graph (-0.0001 / +0.0001). Below the 0.002 rule: **validation and test keep the full-graph CF features**, also for the neural models.
- Side result: with fold-graph features, CF's gain over M1 on all training searches is +0.0026 to +0.0028 instead of +0.0031. The CF selection (E23) therefore depends on a choice of this size, which supports reporting CF as a gain right at the threshold, with M1 alongside (E25).

## 14. Export for the neural models

`05b_neural.ipynb` (Colab GPU) compares its models with LambdaMART on the same searches. It reads the validation scores of the full-data LambdaMART models from `data/processed/neural/lambdamart_validation_scores.parquet` (on Colab: the Drive `processed/neural/` folder), one row per validation candidate in the order of the v3 file.

In [43]:
NEURAL_DIR = PROCESSED / "neural"
NEURAL_DIR.mkdir(parents=True, exist_ok=True)
reference = pq.read_table(feature_file("validation"), columns=["session_id", "wid"]).to_pandas()
for short, key in [("base", "base"), ("m1", "M1"), ("m1_cf", "M1 + CF")]:
    _, _, scores = train_ranker(FINALISTS[key][0], label=F[key], fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
    assert (scores.session_id.to_numpy() == reference.session_id.to_numpy()).all()
    reference[short] = scores.score.to_numpy()
reference.to_parquet(NEURAL_DIR / "lambdamart_validation_scores.parquet", index=False)
reference.describe().loc[["count", "mean", "std"]]

,session_id,wid,base,m1,m1_cf
count,"1,781,573.0000","1,781,573.0000","1,781,573.0000","1,781,573.0000","1,781,573.0000"
mean,"86,762.7850","50,024,754.9298",-1.0919,-1.2194,-1.2939
std,"50,223.1752","28,881,939.4794",1.0057,1.0720,1.1809


## 15. Neural results from 05b (E27-E29)

The neural models are trained on Colab (`05b_neural.ipynb`); their validation scores come back as `data/processed/neural/<name>_validation_scores.parquet`. Here they are evaluated with the same evaluator, compared paired with the final LambdaMART models and logged to the results registry (once per run).

In [44]:
NEURAL_RUNS = {"E27": ("mlp", "feature MLP"), "E28": ("attention", "history attention v1"),
               "E29": ("attention_v2", "history attention v2 (no product IDs)")}
logged = pd.read_csv(RESULTS)
rows = {}
for experiment, (name, label) in NEURAL_RUNS.items():
    score_file, summary_file = NEURAL_DIR / f"{name}_validation_scores.parquet", NEURAL_DIR / f"{name}_summary.json"
    if not (score_file.is_file() and summary_file.is_file()):
        print(f"{experiment}: no results yet ({score_file.name})")
        continue
    scores, summary = pd.read_parquet(score_file), json.loads(summary_file.read_text())
    assert (scores.session_id.to_numpy() == validation_frame.session_id.to_numpy()).all() and not summary["smoke"]
    per_search[label] = evaluate(validation_frame.assign(score=scores.score.to_numpy()), "score",
                                 expected_sizes=validation_sizes)
    for split, ids in [("validation_clean", clean_ids), ("validation", all_ids)]:
        if not ((logged.experiment == experiment) & (logged.model == label) & (logged.split == split)).any():
            log_result(experiment, label, split, per_search[label].loc[ids],
                       notes=(f"05b_neural on Colab ({summary['device']}), {summary['fit_searches']:,} fit searches, "
                              f"best epoch {summary['best_epoch']}, inputs as the final LambdaMART"))
    d_final = paired_difference(per_search[label].loc[clean_ids], per_search[F["M1 + CF"]].loc[clean_ids])
    d_m1 = paired_difference(per_search[label].loc[clean_ids], per_search[F["M1"]].loc[clean_ids])
    rows[label] = {"ndcg_clean": per_search[label].ndcg.loc[clean_ids].mean(), "ndcg_all": per_search[label].ndcg.mean(),
                   "vs_M1_CF": d_final[0], "vs_M1_CF_low": d_final[1], "vs_M1_CF_high": d_final[2],
                   "vs_M1": d_m1[0], "vs_M1_low": d_m1[1], "vs_M1_high": d_m1[2]}
pd.DataFrame(rows).T

,ndcg_clean,ndcg_all,vs_M1_CF,vs_M1_CF_low,vs_M1_CF_high,vs_M1,vs_M1_low,vs_M1_high
feature MLP,0.3889,0.4020,-0.0028,-0.0044,-0.0012,0.0002,-0.0012,0.0019
history attention v1,0.3795,0.3903,-0.0123,-0.0143,-0.0103,-0.0092,-0.0112,-0.0073
history attention v2 (no product IDs),0.3816,0.3935,-0.0101,-0.0122,-0.0081,-0.0071,-0.0090,-0.0050


## 16. E30: checks from the independent reviews (E25)

Four cheap checks, decided before running; none changes the final-model choice (fixed by the pre-registered rule, E23), they decide how the results are worded.

**(a) A fairer untrained rule.** Rule (a) of E17 ranks own products first, then by lagged popularity; the trained base also uses relevance. Here own products come first and the **trained base's score** orders the rest, so the comparison with M1 isolates what learning adds to "known products first".

In [45]:
own_count = pq.read_table(feature_file("validation"), columns=["pooled_product"]).column(0).to_numpy()
_, _, base_all = train_ranker(FINALISTS["base"][0], label=F["base"], fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
b = base_all.score.to_numpy()
RULE_BASE = "rule: own products first, then the trained base"
per_search[RULE_BASE] = evaluate(validation_frame.assign(score=np.nan_to_num(own_count) + 0.999 * (b - b.min()) / (b.max() - b.min())),
                                 "score", expected_sizes=validation_sizes)
display(paired_table([(RULE_BASE, F["base"]), (F["M1"], RULE_BASE), (F["M1 + CF"], RULE_BASE), (RULE_BASE, RULE_A)]))
share = ((per_search[RULE_BASE].ndcg.loc[clean_ids].mean() - per_search[F["base"]].ndcg.loc[clean_ids].mean())
         / (per_search[F["M1"]].ndcg.loc[clean_ids].mean() - per_search[F["base"]].ndcg.loc[clean_ids].mean()))
print(f"share of M1's gain over the base reached by the rule: {share:.1%}")

,A,B,difference,low,high
"rule: own products first, then the trained base minus base, all training",0.3840,0.3500,0.0340,0.0315,0.0365
"M1, all training minus rule: own products first, then the trained base",0.3887,0.3840,0.0047,0.0033,0.0061
"M1 + CF, all training minus rule: own products first, then the trained base",0.3918,0.3840,0.0078,0.0061,0.0094
"rule: own products first, then the trained base minus rule (a): own-product count, then popularity",0.3840,0.3733,0.0107,0.0090,0.0125


share of M1's gain over the base reached by the rule: 87.9%


**(b) Final-gap sensitivity.** Searches whose final gap is 2-5 s behave like the clamped ones (last product in the list 56%, then engaged 66%; E25), and 6-30 s is still elevated. The final models are re-evaluated without them (no retraining), and the E24 safety-gap loss is split by final gap.

In [46]:
gap = searches.final_gap_s.reindex(clean_ids)
populations = {"validation_clean": clean_ids, "without final gap <= 5 s": clean_ids[(gap > 5).to_numpy()],
               "without final gap <= 30 s": clean_ids[(gap > 30).to_numpy()]}
rows = {}
for population, ids in populations.items():
    row = {"searches": len(ids)}
    for short in ("base", "M1", "M1 + CF"):
        row[short] = per_search[F[short]].ndcg.loc[ids].mean()
    d = paired_difference(per_search[F["M1"]].loc[ids], per_search[F["base"]].loc[ids])
    row.update(M1_minus_base=d[0], low=d[1], high=d[2])
    rows[population] = row
display(pd.DataFrame(rows).T)
buckets = pd.cut(gap, [1, 5, 30, 60, 600, 3600, np.inf], labels=["2-5 s", "6-30 s", "31-60 s", "1-10 min", "10-60 min", "over 1 h"])
rows = []
for value in buckets.cat.categories:
    ids = buckets.index[buckets == value]
    d = paired_difference(per_search[M1_CF_GAP].loc[ids], per_search[M1_CF].loc[ids])
    rows.append({"final gap": value, "searches": len(ids), "E24 loss (gap minus normal)": d[0], "low": d[1], "high": d[2],
                 "share of the total loss": d[0] * len(ids) / ((per_search[M1_CF_GAP].ndcg.loc[clean_ids]
                                                               - per_search[M1_CF].ndcg.loc[clean_ids]).sum())})
pd.DataFrame(rows).set_index("final gap")

,searches,base,M1,M1 + CF,M1_minus_base,low,high
validation_clean,"18,844.0000",0.3500,0.3887,0.3918,0.0387,0.0360,0.0415
without final gap <= 5 s,"18,779.0000",0.3500,0.3882,0.3914,0.0382,0.0354,0.0408
without final gap <= 30 s,"17,117.0000",0.3485,0.3814,0.3847,0.0329,0.0303,0.0355


,searches,E24 loss (gap minus normal),low,high,share of the total loss
final gap,,,,,
2-5 s,65,-0.1471,-0.2121,-0.0819,0.0406
6-30 s,1662,-0.0588,-0.0689,-0.0483,0.4147
31-60 s,1818,-0.0377,-0.0470,-0.0292,0.2909
1-10 min,3191,-0.0231,-0.0281,-0.0178,0.3121
10-60 min,955,-0.0005,-0.0054,0.0044,0.0019
over 1 h,11153,0.0013,-0.0002,0.0027,-0.0602


**(c) Training randomness.** LightGBM without subsampling is deterministic, so the bootstrap intervals only cover validation sampling. Here M1 and M1 + CF are retrained on all training searches with `feature_fraction` 0.8 and three seeds; CF minus M1 per seed shows how much of the 0.003 is robust.

In [47]:
seed_rows = []
for seed in (1, 2, 3):
    params = {**PARAMS, "feature_fraction": 0.8, "seed": seed}
    for short in ("M1", "M1 + CF"):
        name = f"{short}, all training, feature_fraction 0.8, seed {seed}"
        _, summary, scores = train_ranker(FINALISTS[short][0], params=params, label=name, fit_sessions=full_fit_ids,
                                          train_splits=ALL_TRAINING)
        per_search[name] = evaluate(scores, "score", expected_sizes=validation_sizes)
        model_summaries[name] = summary
    m1, cf = (f"{s}, all training, feature_fraction 0.8, seed {seed}" for s in ("M1", "M1 + CF"))
    d = paired_difference(per_search[cf].loc[clean_ids], per_search[m1].loc[clean_ids])
    seed_rows.append({"seed": seed, "M1": per_search[m1].ndcg.loc[clean_ids].mean(),
                      "M1 + CF": per_search[cf].ndcg.loc[clean_ids].mean(),
                      "best iterations": f"{model_summaries[m1]['best_iteration']} / {model_summaries[cf]['best_iteration']}",
                      "CF minus M1": d[0], "low": d[1], "high": d[2]})
seeds = pd.DataFrame(seed_rows).set_index("seed")
display(seeds)
print(f"CF minus M1 over seeds: mean {seeds['CF minus M1'].mean():+.4f}, range {seeds['CF minus M1'].min():+.4f} "
      f"to {seeds['CF minus M1'].max():+.4f}; M1 across seeds sd {seeds['M1'].std():.4f}")

[200]	inner's ndcg@10: 0.380819


[400]	inner's ndcg@10: 0.383206


[600]	inner's ndcg@10: 0.384879


[800]	inner's ndcg@10: 0.384718


[200]	inner's ndcg@10: 0.383996


...

[800]	inner's ndcg@10: 0.383901


[200]	inner's ndcg@10: 0.383885


[400]	inner's ndcg@10: 0.385228


,M1,M1 + CF,best iterations,CF minus M1,low,high
seed,,,,,,
1,0.3906,0.3923,717 / 441,0.0017,0.0005,0.0029
2,0.3898,0.3901,541 / 255,0.0004,-0.0009,0.0016
3,0.3906,0.3911,713 / 337,0.0004,-0.0008,0.0016


CF minus M1 over seeds: mean +0.0008, range +0.0004 to +0.0017; M1 across seeds sd 0.0005


**(d) M1 + all six E20 groups on all training searches**, promised for the final runs (PROJECT_PLAN 5) and missing from E23.

In [48]:
M1_ALL_FULL = "M1 + all six groups, all training"
_, summary, scores = train_ranker(E20_MODELS[M1_ALL], label=M1_ALL_FULL, fit_sessions=full_fit_ids, train_splits=ALL_TRAINING)
per_search[M1_ALL_FULL] = evaluate(scores, "score", expected_sizes=validation_sizes)
model_summaries[M1_ALL_FULL] = summary
display(paired_table([(M1_ALL_FULL, F["M1"]), (F["deployable full"], M1_ALL_FULL)]))
for name in [RULE_BASE, M1_ALL_FULL] + [f"{s}, all training, feature_fraction 0.8, seed {seed}"
                                        for seed in (1, 2, 3) for s in ("M1", "M1 + CF")]:
    if not ((pd.read_csv(RESULTS).experiment == "E30") & (pd.read_csv(RESULTS).model == name)).any():
        if name == RULE_BASE:
            log_result("E30", name, "validation_clean", per_search[name].loc[clean_ids],
                       notes="untrained: pooled_product count first, then the all-data base score (features v3)")
        else:
            log_once("E30", name, "validation_clean", per_search[name].loc[clean_ids], model_summaries[name])
pd.read_csv(RESULTS).query("experiment == 'E30'")[["model", "searches", "ndcg_at_10", "commit"]]

[200]	inner's ndcg@10: 0.381925


[400]	inner's ndcg@10: 0.384432


[600]	inner's ndcg@10: 0.385736


,A,B,difference,low,high
"M1 + all six groups, all training minus M1, all training",0.3907,0.3887,0.0020,0.0010,0.0031
"deployable full, all training minus M1 + all six groups, all training",0.3905,0.3907,-0.0001,-0.0011,0.0008


,model,searches,ndcg_at_10,commit
84,"rule: own products first, then the trained base",18844,0.3840,69d5e1a
85,"M1 + all six groups, all training",18844,0.3907,69d5e1a
86,"M1, all training, feature_fraction 0.8, seed 1",18844,0.3906,69d5e1a
87,"M1 + CF, all training, feature_fraction 0.8, s...",18844,0.3923,69d5e1a
88,"M1, all training, feature_fraction 0.8, seed 2",18844,0.3898,69d5e1a
89,"M1 + CF, all training, feature_fraction 0.8, s...",18844,0.3901,69d5e1a
90,"M1, all training, feature_fraction 0.8, seed 3",18844,0.3906,69d5e1a
91,"M1 + CF, all training, feature_fraction 0.8, s...",18844,0.3911,69d5e1a


**Reading the result (E30).**
- **(a)** Own products first, then the trained base: **0.384**, 88% of M1's gain over the base. Learning adds **+0.0047 [0.0033, 0.0061]** (M1) and +0.0078 (M1 + CF) beyond "known products first".
- **(b)** Without gaps up to 5 s nothing changes (+0.038 vs +0.039); **without gaps up to 30 s the personal gain is +0.033 [0.030, 0.036]**. The E24 safety-gap loss comes entirely from searches with a final gap under 10 minutes (41% from 6-30 s). Whether these short gaps carry session evidence or post-search actions cannot be decided; the report gives both numbers.
- **(c)** CF minus M1 across three seeds: +0.0017, +0.0004, +0.0004 (mean +0.0008): **CF is no robust gain**; M1 is an equal co-headline (the pre-registered choice stays). Feature subsampling lifts M1 itself to 0.390-0.391 (sd 0.0005): training randomness is about 0.001-0.002, and mild regularisation helps slightly.
- **(d)** M1 + all six groups with all training data: +0.0020 [0.0010, 0.0031], below the threshold; the full 94-input model adds nothing beyond it (-0.0001).